# 🩺 Diabetic Retinopathy Grading — Production Pipeline **v25 (EffV2-L, 15-day reduced)**

### APTOS 2019 × EyePACS (DR 2015) × DDR → Fine-tune on IDRiD → External validation on Messidor-2

**Target hardware:** HP Victus · Windows 11 · NVIDIA RTX 2050 (4 GB VRAM) · 16 GB RAM · i5-12450H · Jupyter Notebook
**All datasets are local.** No Kaggle API is used anywhere.

---

### ⏱️ 15-day reduced training plan

This variant is calibrated to **finish inside a 15-day window** on a 4 GB GPU with `num_workers=0`.

| Knob | Original v25 | This file | Why |
|---|---|---|---|
| Backbones | 3 (effv2s + effb3 + sex26t) | **1 (EfficientNetV2-L only)** | Single architecture, as requested |
| Seeds | 2 | **1** (`[42]`) | Halves total runs |
| Folds | 5 | **3** | Still produces complete OOF coverage of APTOS |
| P2 epochs | 10 | **6** | Diminishing returns past ep 6 with EMA + cosine |
| Total fold-runs | 30 | **3** | 10× reduction |
| Total epochs | 540 | **42** | (5 P1 + 6 P2 + 3 P3) × 3 folds |

### 🎯 Realistic metrics expectation (NOT 99% — that target is not achievable on APTOS)

| Metric | Realistic | Best case |
|---|---|---|
| OOF QWK (validation) | **0.88 – 0.91** | 0.92 |
| OOF accuracy | **80 – 84%** | 86% |
| Messidor-2 QWK (external) | **0.78 – 0.83** | 0.85 |

> **Why not 99%?** APTOS 2019 has inherent label noise — different ophthalmologists disagree on borderline grades (1 vs 2, 3 vs 4). Inter-rater QWK between human experts is ~0.92. The Kaggle 1st-place winner of 2,943 teams achieved QWK ≈ 0.936. No published method exceeds ~0.94 QWK or ~88% accuracy on APTOS. A model claiming 99% is either evaluated incorrectly, leaking test data, or fabricated.

---

### 🎯 Pipeline strategy (strict execution order)

The notebook is organised so that **every cell depends only on the cells above it**. Run top-to-bottom; each expensive step writes a `*.done` flag so a kernel crash just resumes from the exact point of failure.

| # | Section | Purpose |
|---|---|---|
| 0 | **Setup** | Config · device · imports · checkpoint helpers |
| 1 | **Raw extraction** | Merge & unzip multi-part archives (DDR, Messidor) |
| 2 | **Indexing** | Walk every dataset → build `(path, label, dataset, split)` tables |
| 3 | **Data cleaning** | Fundus verification → blur QC → perceptual-hash dedup → EDA |
| 4 | **Dataset extraction** | Build the **final training dataset** after cleaning is complete: K-fold split + train/val/test |
| 5 | **Modeling primitives** | Augmentations · Dataset · DRModel (GeM + regression head) · EMA · metrics |
| 6 | **Sanity check** | One batch preview before training |
| 7 | **Stage 1 training** | 1 backbone (EfficientNetV2-L) × 1 seed × 3 folds = **3 resumable fold-runs** |
| 8 | **OOF ensemble** | Average fold predictions → optimise thresholds for QWK |
| 9 | **Stage 2 fine-tune** | Low-LR refinement on IDRiD |
| 10 | **External validation** | Messidor-2 (never seen during training) |
| 11 | **Calibration** | Regression → class probabilities |
| 12 | **Inference + Grad-CAM++** | Single-image pipeline with lesion heatmap |
| 13 | **Export** | Ship-ready models + metadata |
| 14 | **Streamlit app** | Deployment UI |
| 15 | **Summary** | Final metrics dump |

### 🧱 Architecture (winner APTOS 2019 strategy, adapted)

- **Backbone (`timm`, ImageNet-21K pretrained):** `tf_efficientnetv2_l.in21k_ft_in1k` (~119M params) — used in every phase, every fold, Stage 1, Stage 2, and external validation
- **Head:** GeM pooling → Linear(256) → BN → ReLU → Dropout(0.5) → Linear(1)
- **Loss:** `SmoothL1Loss` on the regression target (0 – 4); thresholds optimised on OOF for QWK
- **EMA:** decay 0.9999
- **TTA at inference:** original + hflip + vflip averaged

### 💾 Resume guarantee

Every expensive step (extraction, fundus check, blur QC, per-epoch training) writes a `*.done` flag file plus a `.pt` checkpoint. Kernel crash / power cut / reboot → just re-run the cell → work picks up at the exact fold × phase × epoch it left off.

### ⚠️ Operational notes for 15-day run

1. **Plug in the laptop.** RTX 2050 throttles aggressively on battery.
2. **Disable sleep:** `powercfg /change standby-timeout-ac 0` and `powercfg /change monitor-timeout-ac 0` in an Admin cmd.
3. **`num_workers=0`** — mandatory on Windows + Jupyter; parallel workers crash silently.
4. **Verify GPU is computing** with `nvidia-smi` after starting training: GPU-Util > 30%, memory ~2.5–3.5 GB, P0/P2 power state. If GPU-Util = 0% after 5 min, training is stuck — interrupt and check.

---



## 0 · Setup

Single source of truth for paths, hyperparameters, and model list. Edit **only** this section to tune the pipeline.

### 0.1 · Central configuration

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CENTRAL CONFIG — paths, hyperparameters, backbone list
# This is the ONLY cell you should edit to tune the pipeline.
#
# *** v25-EFFV2L 15-DAY REDUCED variant ***
# Single backbone: EfficientNetV2-L (tf_efficientnetv2_l.in21k_ft_in1k, ~119M params).
# Used for ALL phases — Stage 1 training, validation, Stage 2 fine-tune,
# Messidor-2 external test, calibration, and Streamlit deployment.
#
# Reductions vs original v25:
#   SEEDS         [42, 137] → [42]    (halves runs)
#   NUM_FOLDS     5         → 3       (still gives full OOF coverage)
#   P2 epochs     10        → 6       (diminishing returns past ep 6 w/ EMA)
#   Total fold-runs:  30 → 3   (10× cut, fits in 15 days)
#   Total epochs:    540 → 42
# ═══════════════════════════════════════════════════════════════
from pathlib import Path

# ── Dataset root (local, raw downloads live here) ────────────────
DATA_ROOT = Path(r"R:\Dataset")

APTOS_DIR    = DATA_ROOT / "aptos2019"
EYEPACS_DIR  = DATA_ROOT / "dr2015"          # DR 2015 / EyePACS
DDR_DIR      = DATA_ROOT / "DDR Dataset"
IDRID_DIR    = DATA_ROOT / "IDRiD Dataset"
MESSIDOR_DIR = DATA_ROOT / "Messidor Dataset"

# ── Working directories (checkpoints, OOF, logs, exports) ────────
WORK_ROOT  = Path(r"R:\DR_Grading_v24")
CKPT_DIR   = WORK_ROOT / "checkpoints"
OOF_DIR    = WORK_ROOT / "oof"
LOG_DIR    = WORK_ROOT / "logs"
INDEX_DIR  = WORK_ROOT / "indices"
EXPORT_DIR = WORK_ROOT / "export"
FLAG_DIR   = WORK_ROOT / "flags"
EXTRACT_DIR = WORK_ROOT / "extracted"

for d in (CKPT_DIR, OOF_DIR, LOG_DIR, INDEX_DIR,
          EXPORT_DIR, FLAG_DIR, EXTRACT_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ── Training hyperparameters (15-day reduced) ────────────────────
NUM_FOLDS   = 3                    # ↓ from 5 (still covers full APTOS via OOF)
NUM_CLASSES = 5                    # DR grades 0..4
SEEDS       = [42]                 # ↓ from [42, 137]
NUM_WORKERS = 0                    # MANDATORY on Windows + Jupyter (parallel workers crash silently)
PIN_MEMORY  = True

# Total Stage-1 fold-runs = len(MODEL_CONFIGS) × len(SEEDS) × NUM_FOLDS
#                         = 1 × 1 × 3 = 3 fold-runs

# ── Backbone — EfficientNetV2-L ONLY (all phases / folds / stages) ──
# Native pretraining res of V2-L is 384px. We CANNOT use 384px on a 4 GB
# GPU during training, so we run at 192→224px with very small batches and
# heavy gradient accumulation. Effective batch size (batch × accum) is
# kept at ~16 to preserve the optimizer dynamics from the original recipe.
MODEL_CONFIGS = [
    {"name": "effv2l", "backbone": "tf_efficientnetv2_l.in21k_ft_in1k",
     "sz_p1": 192, "sz_p2": 224},
]

# 3-phase training (winner-style progressive resize + unfreeze)
# Batch sizes calibrated for V2-L on RTX 2050 4 GB with AMP enabled.
PHASES = [
    # P1: backbone frozen — only the head trains, so memory is dominated
    # by activations of a forward pass through V2-L. Batch 4 fits comfortably.
    {"name": "P1-Freeze",   "freeze": True,  "epochs": 5, "lr": 3e-4, "batch": 4, "accum": 4},
    # P2: full unfreeze — V2-L gradients + AdamW state push memory hard.
    # Batch 2 is the realistic ceiling at 224px; accum 8 → effective 16.
    # ↓ epochs 10 → 6 (diminishing returns past ep 6 with EMA + cosine LR)
    {"name": "P2-Unfreeze", "freeze": False, "epochs": 6, "lr": 1e-4, "batch": 2, "accum": 8},
    {"name": "P3-Polish",   "freeze": False, "epochs": 3, "lr": 3e-5, "batch": 2, "accum": 8},
]
WD              = 1e-4
EMA_DECAY       = 0.9999
GRAD_CLIP       = 1.0
MIXED_PRECISION = True              # AMP — MANDATORY for V2-L on 4 GB

# Stage 2 (IDRiD fine-tune) — also V2-L, conservative settings
S2_EPOCHS = 6
S2_LR     = 1e-5                    # very low — don't destroy Stage 1 weights
S2_SIZE   = 224                     # match Stage 1 P2 resolution
S2_BATCH  = 2                       # V2-L on 4 GB

# Inference
TTA_FLIPS = True                    # hflip + vflip at test time

# ── Control flags — flip True to force a rerun ───────────────────
FORCE_EXTRACT        = False
FORCE_INDEX          = False
FORCE_FUNDUS_RECHECK = False
FORCE_BLUR_QC        = False
FORCE_STAGE1_RETRAIN = False
FORCE_STAGE2_RETRAIN = False

# ── REALISTIC TIME BUDGET (15 days, V2-L on RTX 2050 4 GB, num_workers=0) ──
#  Per fold (14 epochs total: 5 P1 + 6 P2 + 3 P3):
#    P1-Freeze (5 ep, 192px, batch 4, frozen):     ~ 6–10 hours
#    P2-Unfreeze (6 ep, 224px, batch 2, full):     ~30–48 hours
#    P3-Polish (3 ep, 224px, batch 2, full):       ~12–20 hours
#    Per-fold subtotal:                             ~2–3 days
#  3 folds:                                          ~6–10 days
#  Stage 2 IDRiD fine-tune (6 ep, batch 2):          ~6–10 hours
#  Messidor-2 eval + OOF + calibration + export:     ~2–3 hours
#  Total active compute:                             ~7–11 days
#  + buffer for crashes / restarts / Windows updates: ~3–5 days
#  ⇒ Fits in 15-day window with margin.
#
#  These estimates assume GPU-Util ≥ 30% (verify with nvidia-smi after start).
#  If GPU-Util = 0% after 5 minutes of training → DataLoader stuck, interrupt.

# ── REALISTIC METRICS EXPECTATION ────────────────────────────────
#  OOF QWK     :  0.88 – 0.91   (best case 0.92)
#  OOF accuracy:  80 – 84%      (best case 86%)
#  Messidor-2  :  0.78 – 0.83   QWK
#  ⚠️  99% accuracy/QWK on APTOS is NOT achievable. Best published
#      results: Kaggle 1st place QWK ≈ 0.936, accuracy ≈ 86%.
#      Inter-rater QWK between human ophthalmologists ≈ 0.92.

# ── OOM TROUBLESHOOTING (read if Stage 1 P2 crashes with CUDA OOM) ──
# V2-L on 4 GB is right at the edge. If you hit OOM during P2-Unfreeze:
#   1) Drop sz_p2 to 192 (and sz_p1 to 160).
#   2) If still OOM at batch=2 / 192px: switch the head BatchNorm1d to
#      LayerNorm (in the DRModel definition) and set P2/P3 batch=1, accum=16.
#      BatchNorm1d crashes with batch=1 during training; LayerNorm does not.
#   3) NUM_WORKERS is already 0 (cannot reduce further).

print("✅ Config loaded — 15-DAY REDUCED V2-L plan")
print(f"   Data root   : {DATA_ROOT}")
print(f"   Work root   : {WORK_ROOT}")
print(f"   Backbones   : {[m['backbone'] for m in MODEL_CONFIGS]}")
print(f"   Seeds       : {SEEDS}")
print(f"   Folds       : {NUM_FOLDS}")
print(f"   Fold-runs   : {len(MODEL_CONFIGS)*len(SEEDS)*NUM_FOLDS}  (was 30 in original v25)")
print(f"   Epochs/fold : {sum(p['epochs'] for p in PHASES)}  (5 P1 + 6 P2 + 3 P3)")
print(f"   num_workers : {NUM_WORKERS}  (mandatory 0 on Windows+Jupyter)")


### 0.2 · Device detection

We prefer CUDA, then Apple MPS, then CPU. On the target RTX 2050 (4 GB) mixed-precision is mandatory.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# DEVICE DETECTION + VRAM DIAGNOSTIC
# ═══════════════════════════════════════════════════════════════
import sys, platform
import torch


def detect_device():
    if torch.cuda.is_available():
        dev = torch.device("cuda")
        props = torch.cuda.get_device_properties(0)
        total_gb = props.total_memory / 1024**3
        torch.backends.cudnn.benchmark = True
        print(f"✅ CUDA available — {torch.cuda.get_device_name(0)}")
        print(f"   Compute capability {props.major}.{props.minor}  |  {total_gb:.2f} GB VRAM")
        if total_gb < 6:
            print("   ⚠️  Low VRAM — mixed precision + grad accumulation are ESSENTIAL.")
        return dev, "cuda"
    if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        print("✅ Apple MPS available")
        return torch.device("mps"), "mps"
    print("⚠️  No GPU — falling back to CPU (training will be SLOW)")
    return torch.device("cpu"), "cpu"


DEVICE, DEVICE_KIND = detect_device()
AMP_ENABLED = MIXED_PRECISION and DEVICE_KIND == "cuda"

print(f"\n   Python   : {sys.version.split()[0]}")
print(f"   Torch    : {torch.__version__}")
print(f"   Platform : {platform.system()} {platform.release()}")
print(f"   AMP      : {'ON' if AMP_ENABLED else 'OFF'}")


### 0.3 · Install missing packages

Only installs what is missing, so re-running is cheap.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# INSTALL ONLY MISSING PACKAGES
# ═══════════════════════════════════════════════════════════════
import subprocess, importlib

REQUIRED = {
    # pip name                 : import name
    "timm":                      "timm",
    "albumentations":            "albumentations",
    "opencv-python-headless":    "cv2",
    "scikit-learn":              "sklearn",
    "scikit-image":              "skimage",
    "iterative-stratification":  "iterstrat",
    "tqdm":                      "tqdm",
    "pandas":                    "pandas",
    "pyarrow":                   "pyarrow",
    "matplotlib":                "matplotlib",
    "seaborn":                   "seaborn",
    "pillow":                    "PIL",
}

missing = []
for pip_name, imp_name in REQUIRED.items():
    try:
        importlib.import_module(imp_name)
    except Exception:
        missing.append(pip_name)

if missing:
    print(f"Installing {len(missing)} missing package(s): {missing}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
    print("✅ Installed.")
else:
    print("✅ All required packages already present.")


### 0.4 · Master imports

In [ ]:
# ═══════════════════════════════════════════════════════════════
# MASTER IMPORTS
# ═══════════════════════════════════════════════════════════════
import os, math, time, json, random, shutil, gc, warnings, zipfile
from pathlib import Path
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import cv2
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import Parameter
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    cohen_kappa_score, accuracy_score, f1_score,
    precision_score, recall_score, confusion_matrix,
    classification_report,
)
from sklearn.calibration import calibration_curve

import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)


def seed_everything(seed: int = 42):
    """Make runs reproducible across numpy, python, torch."""
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything(42)
print("✅ Imports OK — timm", timm.__version__, "| albumentations", A.__version__)


### 0.5 · Resume & checkpoint helpers

Every expensive step writes a `*.done` flag to `FLAG_DIR`. Atomic save ensures a half-written checkpoint never corrupts a resume.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# RESUME / CHECKPOINT HELPERS
# ═══════════════════════════════════════════════════════════════
def flag_path(tag: str) -> Path:
    return FLAG_DIR / f"{tag}.done"


def is_done(tag: str) -> bool:
    return flag_path(tag).exists()


def mark_done(tag: str):
    flag_path(tag).write_text(time.strftime("%Y-%m-%d %H:%M:%S"))


def clear_done(tag: str):
    fp = flag_path(tag)
    if fp.exists():
        fp.unlink()


def safe_load(path: Path, map_location="cpu"):
    """Load a torch checkpoint with retry on partial write."""
    for attempt in range(3):
        try:
            return torch.load(path, map_location=map_location)
        except Exception:
            if attempt == 2:
                raise
            time.sleep(1)


def safe_save(obj, path: Path):
    """Atomic save: write to .tmp, then rename."""
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(obj, tmp)
    tmp.replace(path)


def step_header(n, title):
    bar = "═" * 64
    print(f"\n{bar}\n  STEP {n} — {title}\n{bar}")
    return time.time()


def step_skip(n, title, reason):
    print(f"  [SKIP] STEP {n} — {title} — {reason}")


def step_done(t0):
    print(f"  ⏱  {time.time() - t0:.1f}s")


print("✅ Resume system ready. Flag dir:", FLAG_DIR)
existing = [f.stem for f in FLAG_DIR.glob("*.done")]
print("   Existing flags:", existing or "(none)")


## 1 · Raw archive extraction

**Why this runs first:** indexing (Section 2) walks the filesystem and builds path tables. If a dataset is still sitting as `DDR-dataset.zip.001 / .002 / …`, those files don't physically exist on disk yet, so we extract here first.

Handles both common split-archive conventions:
- WinRAR-style: `foo.zip`, `foo.zip.002`, `foo.zip.003`, …
- Classic split: `foo.zip.001`, `foo.zip.002`, …

APTOS, EyePACS (dr2015), and IDRiD are expected to already be un-archived in their folders.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# MULTI-PART ZIP → MERGE → EXTRACT UTILITY
# ═══════════════════════════════════════════════════════════════
def find_parts(base_dir: Path, stem: str):
    """
    Find ordered parts of a split archive.
    Supports:
      (A) stem.zip.001, stem.zip.002, ...         ← first part is .zip.001
      (B) stem.zip,     stem.zip.002, ...         ← WinRAR style
    Returns [] if nothing matches.
    """
    parts = []

    # Case (A): parts start at .zip.001
    p001 = base_dir / f"{stem}.zip.001"
    if p001.exists():
        i = 1
        while True:
            p = base_dir / f"{stem}.zip.{i:03d}"
            if not p.exists():
                break
            parts.append(p)
            i += 1
        return parts

    # Case (B): plain .zip + .zip.002 / .zip.003 / ...
    p0 = base_dir / f"{stem}.zip"
    if p0.exists():
        parts.append(p0)
        i = 2
        while True:
            p = base_dir / f"{stem}.zip.{i:03d}"
            if not p.exists():
                break
            parts.append(p)
            i += 1
        return parts

    return parts


def merge_and_extract(parts, out_dir: Path, tag: str, delete_merged: bool = True):
    """
    Concatenate parts → one .zip → extract to out_dir.
    Skips if `tag` is already marked done.
    """
    if is_done(tag) and not FORCE_EXTRACT:
        step_skip("1", f"extract {tag}", "already extracted")
        return out_dir

    if not parts:
        print(f"  ⚠️  No parts supplied for {tag} — skipping.")
        return out_dir

    out_dir.mkdir(parents=True, exist_ok=True)
    total_mb = sum(p.stat().st_size for p in parts) / 1024**2
    print(f"  {tag}: {len(parts)} part(s), {total_mb:,.0f} MB total")
    for p in parts:
        print(f"     • {p.name}  ({p.stat().st_size/1024**2:,.1f} MB)")

    # Single file? Extract directly.
    if len(parts) == 1:
        with zipfile.ZipFile(parts[0]) as zf:
            for m in tqdm(zf.infolist(), desc=f"extract {tag}", leave=False):
                try:
                    zf.extract(m, out_dir)
                except Exception as e:
                    print(f"     ! skip {m.filename}: {e}")
        mark_done(tag)
        return out_dir

    # Split archive — merge then extract.
    merge_tmp_dir = WORK_ROOT / "_merge_tmp"
    merge_tmp_dir.mkdir(parents=True, exist_ok=True)
    merged = merge_tmp_dir / f"{tag}_merged.zip"

    print(f"  Merging {len(parts)} parts → {merged.name} ...")
    with open(merged, "wb") as fout:
        for p in tqdm(parts, desc="merge", leave=False):
            with open(p, "rb") as fin:
                shutil.copyfileobj(fin, fout, length=16 * 1024 * 1024)

    print(f"  Extracting merged archive → {out_dir} ...")
    try:
        with zipfile.ZipFile(merged) as zf:
            for m in tqdm(zf.infolist(), desc="extract", leave=False):
                try:
                    zf.extract(m, out_dir)
                except Exception as e:
                    print(f"     ! skip {m.filename}: {e}")
    finally:
        try:
            merged.unlink()
            merge_tmp_dir.rmdir()
        except Exception:
            pass

    mark_done(tag)
    print(f"  ✅ Extracted → {out_dir}")
    return out_dir


print("✅ Extraction utility ready.")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# EXTRACT DDR + MESSIDOR multi-part archives
# APTOS / EyePACS / IDRiD are expected to be pre-extracted.
# ═══════════════════════════════════════════════════════════════
t0 = step_header(1, "RAW ARCHIVE EXTRACTION")

# Where extracted payloads will live
DDR_EX_DIR  = EXTRACT_DIR / "ddr"
MESS_EX_DIR = EXTRACT_DIR / "messidor"

# ── DDR: parts are DDR-dataset.zip.001 … .010 ─────────────────
ddr_parts = find_parts(DDR_DIR, "DDR-dataset")
if ddr_parts:
    print(f"\nDDR: found {len(ddr_parts)} part(s)")
    merge_and_extract(ddr_parts, DDR_EX_DIR, tag="extract_ddr")
else:
    print(f"\nDDR: no split archive found in {DDR_DIR} "
          f"(assuming already extracted)")

# ── Messidor-2: parts are IMAGES.zip.001 … .00N ──────────────
mess_parts = find_parts(MESSIDOR_DIR, "IMAGES")
if mess_parts:
    print(f"\nMessidor: found {len(mess_parts)} part(s)")
    merge_and_extract(mess_parts, MESS_EX_DIR, tag="extract_messidor")
else:
    print(f"\nMessidor: no split archive found in {MESSIDOR_DIR} "
          f"(assuming already extracted)")

# ── Sanity count of extracted images ──────────────────────────
def _count_images(d: Path) -> int:
    if not d.exists():
        return 0
    n = 0
    for ext in ("*.png", "*.jpg", "*.jpeg", "*.tif", "*.tiff"):
        n += sum(1 for _ in d.rglob(ext))
    return n


print(f"\n  DDR extracted images     : {_count_images(DDR_EX_DIR):>8,}  @ {DDR_EX_DIR}")
print(f"  Messidor extracted images: {_count_images(MESS_EX_DIR):>8,}  @ {MESS_EX_DIR}")
print(f"  APTOS images (raw)       : {_count_images(APTOS_DIR):>8,}  @ {APTOS_DIR}")
print(f"  EyePACS images (raw)     : {_count_images(EYEPACS_DIR):>8,}  @ {EYEPACS_DIR}")
print(f"  IDRiD images (raw)       : {_count_images(IDRID_DIR):>8,}  @ {IDRID_DIR}")
step_done(t0)


## 2 · Build a unified image index

For every dataset we produce a parquet with schema `(img_path, label, dataset, split)`. Later cells never touch raw folders — they go through these parquets.

All index-builders are **idempotent**: they re-read existing parquets on re-run unless you set `FORCE_INDEX = True` in Section 0.1.

### 2.1 · APTOS 2019 index

`train.csv` with columns `id_code, diagnosis` + `train_images/` folder.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# INDEX: APTOS 2019
# ═══════════════════════════════════════════════════════════════
t0 = step_header("2.1", "INDEX APTOS 2019")

aptos_parquet   = INDEX_DIR / "aptos.parquet"
aptos_train_csv = APTOS_DIR / "train.csv"
aptos_img_dir   = APTOS_DIR / "train_images"

if aptos_parquet.exists() and not FORCE_INDEX:
    df_ap = pd.read_parquet(aptos_parquet)
    print(f"  [cached] APTOS: {len(df_ap):,} images")
elif aptos_train_csv.exists() and aptos_img_dir.exists():
    df_ap = pd.read_csv(aptos_train_csv)

    def _ap_path(idc):
        for ext in (".png", ".jpg", ".jpeg"):
            p = aptos_img_dir / f"{idc}{ext}"
            if p.exists():
                return str(p)
        return None

    df_ap["img_path"] = df_ap["id_code"].map(_ap_path)
    df_ap = (df_ap.dropna(subset=["img_path"])
                   .rename(columns={"diagnosis": "label"})
                   [["img_path", "label"]])
    df_ap["dataset"] = "aptos"
    df_ap["split"]   = "train"
    df_ap.to_parquet(aptos_parquet, index=False)
    print(f"  ✅ APTOS: {len(df_ap):,} images saved → {aptos_parquet.name}")
    print(df_ap["label"].value_counts().sort_index().to_string())
else:
    df_ap = pd.DataFrame(columns=["img_path", "label", "dataset", "split"])
    print(f"  ⚠️  APTOS not found at {APTOS_DIR}")

step_done(t0)


### 2.2 · EyePACS / DR 2015 index

EyePACS ships in multiple formats — plain CSV, class-subfolder layout, or both. We try each strategy in turn.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# INDEX: EyePACS / DR 2015
# ═══════════════════════════════════════════════════════════════
t0 = step_header("2.2", "INDEX EyePACS / DR 2015")

eyepacs_parquet = INDEX_DIR / "eyepacs.parquet"


def build_eyepacs_index(root: Path) -> pd.DataFrame:
    """Try CSV labels first, fall back to class-subfolder layout."""
    if not root.exists():
        print(f"  ⚠️  Not found: {root}")
        return pd.DataFrame(columns=["img_path", "label", "dataset", "split"])

    # Build name→path map (stems, full names, lowercase variants)
    print(f"  Scanning all images under {root} ...")
    name2path = {}
    for ext in ("*.jpeg", "*.jpg", "*.png"):
        for p in root.rglob(ext):
            name2path[p.stem] = str(p)
            name2path[p.name] = str(p)
            name2path[p.stem + ".jpeg"] = str(p)
            name2path[p.stem + ".jpg"]  = str(p)
    total_imgs = len({v for v in name2path.values()})
    print(f"  Images on disk: {total_imgs:,}")

    if total_imgs == 0:
        print("  ❌ Zero images on disk — dr2015/ appears unextracted.")
        return pd.DataFrame(columns=["img_path", "label", "dataset", "split"])

    rows = []

    # Strategy A — any CSV with image + label columns
    for csv_p in list(root.rglob("*.csv")):
        try:
            df_tmp = None
            for sep in (",", "\t", ";", " "):
                try:
                    df_t = pd.read_csv(csv_p, sep=sep)
                    if len(df_t.columns) >= 2:
                        df_tmp = df_t
                        break
                except Exception:
                    continue
            if df_tmp is None:
                continue
            df_tmp.columns = [c.strip() for c in df_tmp.columns]

            img_col = next((c for c in df_tmp.columns
                            if c.lower() in ("image", "id_code", "filename", "name", "image_name")), None)
            lbl_col = next((c for c in df_tmp.columns
                            if c.lower() in ("level", "diagnosis", "label", "grade", "retinopathy_grade")), None)

            # Auto-detect image col by regex if needed
            if img_col is None:
                for c in df_tmp.columns:
                    try:
                        s = df_tmp[c].dropna().astype(str).head(5)
                        if s.str.match(r"^[a-zA-Z0-9_\-\.]+$").mean() > 0.7:
                            img_col = c
                            break
                    except Exception:
                        pass

            # Auto-detect label col: numeric and in [0,4]
            if lbl_col is None:
                for c in df_tmp.columns:
                    if c == img_col:
                        continue
                    try:
                        vals = pd.to_numeric(df_tmp[c], errors="coerce").dropna()
                        if len(vals) > 10 and vals.between(0, 4).mean() > 0.95:
                            lbl_col = c
                            break
                    except Exception:
                        pass

            if img_col is None or lbl_col is None:
                continue

            split_name = "test" if "test" in csv_p.stem.lower() else "train"
            for _, r in df_tmp.iterrows():
                key = str(r[img_col]).strip()
                path = (name2path.get(key)
                        or name2path.get(Path(key).stem)
                        or name2path.get(Path(key).stem + ".jpeg")
                        or name2path.get(Path(key).stem + ".jpg"))
                if path is None:
                    continue
                try:
                    lbl = int(r[lbl_col])
                except Exception:
                    continue
                if 0 <= lbl <= 4:
                    rows.append({"img_path": path, "label": lbl,
                                 "dataset": "eyepacs", "split": split_name})
            print(f"  ✓ CSV {csv_p.name}: {len(rows):,} rows matched so far")
        except Exception as e:
            print(f"  ! {csv_p.name}: {e}")
            continue

    # Strategy B — class subfolder layout 0/1/2/3/4
    if not rows:
        print("  CSV strategy failed; trying class-subfolder layout ...")
        for sub in root.rglob("*"):
            if not sub.is_dir():
                continue
            try:
                children = {c.name for c in sub.iterdir() if c.is_dir()}
            except Exception:
                continue
            if not {"0", "1", "2", "3", "4"}.issubset(children):
                continue
            split_name = sub.name.lower()
            for lbl_str in ("0", "1", "2", "3", "4"):
                for ext in ("*.jpeg", "*.jpg", "*.png"):
                    for p in (sub / lbl_str).rglob(ext):
                        rows.append({"img_path": str(p), "label": int(lbl_str),
                                     "dataset": "eyepacs", "split": split_name})

    return pd.DataFrame(rows)


if eyepacs_parquet.exists() and not FORCE_INDEX:
    df_ep = pd.read_parquet(eyepacs_parquet)
    print(f"  [cached] EyePACS: {len(df_ep):,} images")
else:
    df_ep = build_eyepacs_index(EYEPACS_DIR)
    if len(df_ep):
        df_ep.to_parquet(eyepacs_parquet, index=False)
        print(f"\n  ✅ EyePACS: {len(df_ep):,} images "
              f"splits={df_ep['split'].value_counts().to_dict()}")
        print(df_ep["label"].value_counts().sort_index().to_string())
    else:
        print("  ℹ️  EyePACS empty — pipeline continues without it.")

step_done(t0)


### 2.3 · DDR index

DDR typically ships with class-subfolder layout `{train,valid,test}/{0,1,2,3,4,5}/`. Grade 5 = ungradable → dropped.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# INDEX: DDR (folder-labelled, drops ungradable grade-5)
# ═══════════════════════════════════════════════════════════════
t0 = step_header("2.3", "INDEX DDR")

ddr_parquet = INDEX_DIR / "ddr.parquet"


def build_ddr_index(root: Path) -> pd.DataFrame:
    if not root.exists():
        return pd.DataFrame(columns=["img_path", "label", "dataset", "split"])

    # Find any folder whose children are 0,1,2,3,4 (and optionally 5)
    hits = []
    for sub in root.rglob("*"):
        if not sub.is_dir():
            continue
        try:
            child_names = {c.name for c in sub.iterdir() if c.is_dir()}
        except Exception:
            continue
        if not child_names.issuperset({"0", "1", "2", "3", "4"}):
            continue
        split_name = sub.name.lower()
        for lbl in ("0", "1", "2", "3", "4", "5"):
            cls_dir = sub / lbl
            if not cls_dir.exists():
                continue
            for ext in ("*.jpg", "*.jpeg", "*.png"):
                for p in cls_dir.rglob(ext):
                    hits.append({
                        "img_path": str(p),
                        "label":    int(lbl),
                        "dataset":  "ddr",
                        "split":    split_name,
                    })

    # Fallback: csv/txt with "filename label"
    if not hits:
        for txt in list(root.rglob("*.txt")) + list(root.rglob("*.csv")):
            try:
                tmp = pd.read_csv(txt, sep=None, engine="python",
                                  header=None, names=["image", "label"])
                name2path = {p.stem: str(p) for ext in ("*.jpg", "*.jpeg", "*.png")
                             for p in root.rglob(ext)}
                tmp["img_path"] = tmp["image"].astype(str).map(
                    lambda s: name2path.get(Path(s).stem))
                tmp = tmp.dropna(subset=["img_path"])[["img_path", "label"]]
                tmp["dataset"] = "ddr"
                tmp["split"]   = txt.stem.lower()
                hits.extend(tmp.to_dict("records"))
            except Exception:
                continue

    df = pd.DataFrame(hits)
    if len(df):
        df["label"] = df["label"].astype(int)
        df = df[df["label"].between(0, 4)].reset_index(drop=True)   # drop grade-5 ungradable
    return df


if ddr_parquet.exists() and not FORCE_INDEX:
    df_ddr = pd.read_parquet(ddr_parquet)
    print(f"  [cached] DDR: {len(df_ddr):,} images")
else:
    df_ddr = build_ddr_index(DDR_EX_DIR)
    if len(df_ddr):
        df_ddr.to_parquet(ddr_parquet, index=False)
        print(f"  ✅ DDR: {len(df_ddr):,} images  "
              f"splits={df_ddr['split'].unique().tolist()}")
        print(df_ddr["label"].value_counts().sort_index().to_string())
    else:
        print("  ⚠️  DDR index empty — check extraction / folder structure.")

step_done(t0)


### 2.4 · IDRiD index (Disease-Grading subset)

Labels live in `2. Groundtruths/a. IDRiD_Disease Grading_{Train,Test}ing Labels.csv`.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# INDEX: IDRiD (Disease Grading subset)
# ═══════════════════════════════════════════════════════════════
t0 = step_header("2.4", "INDEX IDRiD")

idrid_parquet = INDEX_DIR / "idrid.parquet"


def build_idrid_index(root: Path) -> pd.DataFrame:
    if not root.exists():
        return pd.DataFrame(columns=["img_path", "label", "dataset", "split"])

    csv_train = csv_test = None
    for p in root.rglob("*.csv"):
        n = p.name.lower()
        if "disease" in n and "grad" in n:
            if "train" in n:
                csv_train = p
            elif "test" in n:
                csv_test = p

    # Build filename→path map once
    name2path = {}
    for ext in ("*.jpg", "*.jpeg", "*.png"):
        for p in root.rglob(ext):
            name2path[p.stem] = str(p)

    rows = []

    def _ingest(csv_path, split_name):
        if csv_path is None or not csv_path.exists():
            return
        df = pd.read_csv(csv_path)
        df.columns = [c.strip() for c in df.columns]
        img_col = next((c for c in df.columns if "image" in c.lower()), None)
        grd_col = next((c for c in df.columns if "retin" in c.lower()), None)
        if img_col is None or grd_col is None:
            return
        for _, r in df.iterrows():
            stem = str(r[img_col]).strip()
            path = name2path.get(stem) or name2path.get(Path(stem).stem)
            if path is None:
                continue
            try:
                lbl = int(r[grd_col])
            except Exception:
                continue
            if 0 <= lbl <= 4:
                rows.append({"img_path": path, "label": lbl,
                             "dataset": "idrid", "split": split_name})

    _ingest(csv_train, "train")
    _ingest(csv_test,  "test")
    return pd.DataFrame(rows)


if idrid_parquet.exists() and not FORCE_INDEX:
    df_idrid = pd.read_parquet(idrid_parquet)
    print(f"  [cached] IDRiD: {len(df_idrid):,} images")
else:
    df_idrid = build_idrid_index(IDRID_DIR)
    if len(df_idrid):
        df_idrid.to_parquet(idrid_parquet, index=False)
        print(f"  ✅ IDRiD: {len(df_idrid):,} images  "
              f"splits={df_idrid['split'].value_counts().to_dict()}")
        print(df_idrid["label"].value_counts().sort_index().to_string())
    else:
        print("  ⚠️  IDRiD index empty — check path + ground-truth CSV.")

step_done(t0)


### 2.5 · Messidor-2 index

The official Messidor-2 CSV uses Google-Brain adjudicated 0–4 grades, no remapping needed. The CSV is **comma-separated** — auto-detecting the separator historically caused 40 % of the grades to silently merge with the `image_id` field, so we force `sep=","` and re-verify counts against the published reference `(1017, 270, 347, 75, 35)`.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# INDEX: Messidor-2 (external validation set)
# ═══════════════════════════════════════════════════════════════
t0 = step_header("2.5", "INDEX Messidor-2")

messidor_parquet = INDEX_DIR / "messidor.parquet"

EXPECTED_MESSIDOR = {0: 1017, 1: 270, 2: 347, 3: 75, 4: 35}   # per the paper


def build_messidor_index(mess_dir: Path, ex_dir: Path) -> pd.DataFrame:
    grade_csv = mess_dir / "messidor_data.csv"
    if not grade_csv.exists():
        raise FileNotFoundError(f"messidor_data.csv not found at {grade_csv}")

    # Force comma separator — auto-detection has silently corrupted this before.
    df_g = pd.read_csv(grade_csv, sep=",", dtype=str)
    df_g.columns = [c.strip() for c in df_g.columns]
    if len(df_g.columns) == 1:
        raise ValueError(
            "Messidor CSV parsed as 1 column — separator is wrong. "
            "Inspect the file manually.")

    img_col = "image_id" if "image_id" in df_g.columns else next(
        (c for c in df_g.columns
         if any(k in c.lower() for k in ("image", "id", "file", "name"))), None)
    grd_col = "adjudicated_dr_grade" if "adjudicated_dr_grade" in df_g.columns else next(
        (c for c in df_g.columns
         if any(k in c.lower() for k in ("dr_grade", "grade", "level"))), None)

    # Convert grades: strip, to int, drop NaN
    df_g["image_id_clean"] = df_g[img_col].astype(str).str.strip()
    df_g["grade_int"] = pd.to_numeric(df_g[grd_col].str.strip(), errors="coerce")
    n_before = len(df_g)
    df_g = df_g.dropna(subset=["grade_int"]).copy()
    df_g["grade_int"] = df_g["grade_int"].astype(int)
    print(f"  CSV rows: {n_before} → {len(df_g)} after dropping ungradable")

    # Build a forgiving lookup: original, lowercase, stem, and stem+ext variants
    grade_map = {}
    for _, row in df_g.iterrows():
        raw = row["image_id_clean"]
        lbl = int(row["grade_int"])
        stem = Path(raw).stem
        for variant in (raw, raw.lower(), stem, stem.lower(),
                        stem + ".png", stem + ".jpg",
                        stem + ".jpeg", stem + ".tif",
                        stem.lower() + ".png", stem.lower() + ".jpg"):
            grade_map[variant] = lbl

    # Scan every plausible extracted location
    search_roots = [
        ex_dir / "IMAGES", ex_dir,
        mess_dir / "IMAGES", mess_dir,
    ]
    name2path = {}
    for root in search_roots:
        if not root.exists():
            continue
        for ext in ("*.png", "*.jpg", "*.jpeg", "*.tif", "*.tiff"):
            for p in root.rglob(ext):
                name2path[p.name]               = str(p)
                name2path[p.name.lower()]       = str(p)
                name2path[p.stem]               = str(p)
                name2path[p.stem.lower()]       = str(p)

    unique_disk = len({v for v in name2path.values()})
    print(f"  Unique images on disk: {unique_disk:,}")

    # Match filenames → grades
    rows, no_grade, seen = [], [], set()
    for fname, fpath in name2path.items():
        if "." not in fname or fpath in seen:
            continue
        lbl = (grade_map.get(fname) or grade_map.get(fname.lower())
               or grade_map.get(Path(fname).stem)
               or grade_map.get(Path(fname).stem.lower()))
        if lbl is None:
            no_grade.append(fname)
            continue
        rows.append({"img_path": fpath, "label": int(lbl),
                     "dataset": "messidor", "split": "external"})
        seen.add(fpath)

    df_out = (pd.DataFrame(rows).drop_duplicates("img_path").reset_index(drop=True)
              if rows else
              pd.DataFrame(columns=["img_path", "label", "dataset", "split"]))
    print(f"  Matched: {len(df_out):,}   No-grade: {len(no_grade):,}")
    return df_out


if messidor_parquet.exists() and not FORCE_INDEX:
    df_mss = pd.read_parquet(messidor_parquet)
    print(f"  [cached] Messidor-2: {len(df_mss):,} images")
else:
    df_mss = build_messidor_index(MESSIDOR_DIR, MESS_EX_DIR)
    if len(df_mss):
        df_mss.to_parquet(messidor_parquet, index=False)
        actual = df_mss["label"].value_counts().sort_index().to_dict()
        print(f"\n  ✅ Messidor-2: {len(df_mss):,} images saved → {messidor_parquet.name}")
        print(f"\n  {'Grade':<8} {'Expected':>10} {'Got':>10} {'OK?':>8}")
        for g, exp in EXPECTED_MESSIDOR.items():
            got = actual.get(g, 0)
            ok = "✅" if got == exp else f"⚠️ {got-exp:+d}"
            print(f"  {g:<8} {exp:>10,} {got:>10,} {ok:>8}")
    else:
        print("  ❌ Messidor-2 index is empty — check extraction + CSV.")

step_done(t0)


### 2.6 · Summary of raw indices

One-line view of how many images each dataset contributed before any cleaning.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# INDEX SUMMARY
# ═══════════════════════════════════════════════════════════════
t0 = step_header("2.6", "INDEX SUMMARY")

summary_rows = []
for name, df in [("APTOS", df_ap), ("EyePACS", df_ep), ("DDR", df_ddr),
                 ("IDRiD", df_idrid), ("Messidor-2", df_mss)]:
    row = {"dataset": name, "n_images": len(df)}
    for cls in range(5):
        row[f"class_{cls}"] = int((df["label"] == cls).sum()) if len(df) else 0
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))
summary_df.to_csv(LOG_DIR / "dataset_summary_raw.csv", index=False)
step_done(t0)


## 3 · Data cleaning

All datasets are cleaned in a single mandatory pass before the training pool is assembled. Three stages, each result cached as a parquet so no work is ever repeated:

1. **Blur QC + perceptual-hash deduplication** (§3.1) — drops the bottom-5 % blurriest images and exact pHash duplicates across every dataset.
2. **Exploratory data analysis** (§3.2) — class balance, blur-by-grade, and QC acceptance rate plots saved to `LOG_DIR`.
3. **Fundus validator training** (§3.3) — trains a MobileNetV3-Small binary classifier on cleaned positives + synthetic negatives. This model is **not** used to filter training data. It is exported to `EXPORT_DIR/fundus_validator.pt` and loaded by the Streamlit app at runtime to reject non-fundus user uploads before grading.

> **Execution order guarantee:** Section 4 (training pool assembly) runs only after §3.1 and §3.2 complete, ensuring the final pool contains zero blurry or duplicate images.

### 3.1 · Blur QC + perceptual-hash deduplication

Applied directly to every raw indexed dataset — no prior fundus gating:

- **Laplacian variance** on a 256×256 grayscale downsample → blurriness proxy (lower = blurrier). Bottom 5 % per dataset are dropped.
- **8×8 DCT-based pHash** → exact/near-duplicate detector. First occurrence kept, rest dropped.

Output: `indices/{name}_clean.parquet` — the sole input to Section 4.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# STEP 3.1 — BLUR QC + PERCEPTUAL-HASH DEDUPLICATION
# -----------------------------------------------------------------------
# Input : raw indexed DataFrames (df_ap, df_ep, df_ddr, df_idrid, df_mss)
# Output: indices/{name}_clean.parquet  — used by Section 4 exclusively
#
# No fundus-gate filter is applied here. Every image in the index is
# evaluated. Non-fundus images that slip past are handled at inference
# time by the trained fundus validator (§3.3) in the Streamlit app.
# ═══════════════════════════════════════════════════════════════════════

def blur_and_hash(path: str) -> tuple:
    """Return (laplacian_variance, phash_binary_string) for a single image."""
    try:
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            return (0.0, "")
        small = cv2.resize(img, (256, 256), interpolation=cv2.INTER_AREA)
        lap_var = float(cv2.Laplacian(small, cv2.CV_64F).var())
        h32  = cv2.resize(img, (32, 32), interpolation=cv2.INTER_AREA).astype(np.float32)
        dct  = cv2.dct(h32)[:8, :8]
        bits = (dct > np.median(dct)).flatten().astype(np.uint8)
        return (lap_var, "".join(map(str, bits)))
    except Exception:
        return (0.0, "")


def qc_dataset(df: pd.DataFrame, name: str,
               blur_pct: float = 5.0, workers: int = 12) -> pd.DataFrame:
    """
    Drop bottom `blur_pct` % blurriest images and exact pHash duplicates.

    Parameters
    ----------
    df       : raw indexed DataFrame with an `img_path` column.
    name     : dataset name used for cache file naming.
    blur_pct : percentile cut-off (default 5 → bottom 5 % removed).
    workers  : ThreadPoolExecutor concurrency.

    Returns
    -------
    Cleaned DataFrame with added `laplacian` and `phash` columns.
    Saved to `INDEX_DIR/{name}_clean.parquet`.
    """
    out_path = INDEX_DIR / f"{name}_clean.parquet"
    if out_path.exists() and not FORCE_BLUR_QC:
        print(f"  ✅ {name}: loaded from cache → {out_path.name}", flush=True)
        return pd.read_parquet(out_path)

    df_in = df.reset_index(drop=True).copy()
    n = len(df_in)
    if n == 0:
        df_in["laplacian"] = pd.Series(dtype=float)
        df_in["phash"]     = pd.Series(dtype=str)
        df_in.to_parquet(out_path, index=False)
        return df_in

    print(f"  {name}: computing blur+pHash on {n:,} images (workers={workers})",
          flush=True)
    t0 = time.time()
    results = [None] * n

    with ThreadPoolExecutor(max_workers=workers) as ex:
        futs = {ex.submit(blur_and_hash, p): i
                for i, p in enumerate(df_in["img_path"])}
        done, last = 0, t0
        for fut in as_completed(futs):
            results[futs[fut]] = fut.result()
            done += 1
            now = time.time()
            if done % 500 == 0 or (now - last) >= 5:
                rate = done / max(1e-6, now - t0)
                eta  = (n - done) / max(1e-6, rate) / 60
                print(f"    [{name}] {done:,}/{n:,} ({done / n * 100:.1f}%)"
                      f"  {rate:.0f} img/s  ETA {eta:.1f}m", flush=True)
                last = now

    df_in["laplacian"] = [r[0] for r in results]
    df_in["phash"]     = [r[1] for r in results]

    # ── Blur filter ───────────────────────────────────────────────────
    n_raw    = len(df_in)
    cutoff   = np.percentile(df_in["laplacian"], blur_pct)
    df_in    = df_in[df_in["laplacian"] >= cutoff].reset_index(drop=True)
    n_blur   = len(df_in)

    # ── Duplicate filter ──────────────────────────────────────────────
    df_in    = df_in.drop_duplicates(subset=["phash"], keep="first").reset_index(drop=True)
    n_clean  = len(df_in)

    df_in.to_parquet(out_path, index=False)
    elapsed = time.time() - t0
    print(f"  ✅ {name}: {n_raw:,} → blur-drop → {n_blur:,} "
          f"→ dedup → {n_clean:,}  ({elapsed / 60:.2f}m)")
    return df_in


# ── Apply to every dataset ────────────────────────────────────────────
t0 = step_header("3.1", "BLUR QC + DEDUPLICATION — ALL DATASETS")

df_ap_c    = qc_dataset(df_ap,    "aptos")
df_ep_c    = qc_dataset(df_ep,    "eyepacs")
df_ddr_c   = qc_dataset(df_ddr,   "ddr")
df_idrid_c = qc_dataset(df_idrid, "idrid")
df_mss_c   = qc_dataset(df_mss,   "messidor")

step_done(t0)

# ── Cleaning summary ──────────────────────────────────────────────────
print("\n  Cleaning summary (blur + dedup):")
print(f"  {'Dataset':<12} {'Before':>8} {'After':>8} {'Dropped':>8} {'Kept%':>7}")
print("  " + "─" * 47)
for name, df_raw, df_c in [
    ("aptos",    df_ap,    df_ap_c),
    ("eyepacs",  df_ep,    df_ep_c),
    ("ddr",      df_ddr,   df_ddr_c),
    ("idrid",    df_idrid, df_idrid_c),
    ("messidor", df_mss,   df_mss_c),
]:
    before  = len(df_raw)
    after   = len(df_c)
    dropped = before - after
    pct     = after / max(1, before) * 100
    print(f"  {name:<12} {before:>8,} {after:>8,} {dropped:>8,} {pct:>6.1f}%")

total_before = sum(len(d) for d in [df_ap, df_ep, df_ddr, df_idrid, df_mss])
total_after  = sum(len(d) for d in [df_ap_c, df_ep_c, df_ddr_c, df_idrid_c, df_mss_c])
print(f"  {'TOTAL':<12} {total_before:>8,} {total_after:>8,} "
      f"{total_before - total_after:>8,} {total_after / max(1, total_before) * 100:>6.1f}%")
print()
print("✅ Step 3.1 complete — all datasets cleaned.")


### 3.2 · Exploratory data analysis

Three diagnostic plots saved to `LOG_DIR` as PNGs:

1. Per-dataset class distribution (after cleaning)
2. Blur score distribution per DR grade
3. QC acceptance rate per dataset (before vs after blur+dedup)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 3.2 — EDA: class distribution, blur-by-grade, QC acceptance rates
# ═══════════════════════════════════════════════════════════════
GRADE_NAMES = ["0-None", "1-Mild", "2-Moderate", "3-Severe", "4-Proliferative"]
GRADE_COLORS = ["#2ecc71", "#f1c40f", "#e67e22", "#e74c3c", "#8e44ad"]

datasets_eda = [("aptos", df_ap_c), ("eyepacs", df_ep_c), ("ddr", df_ddr_c),
                ("idrid", df_idrid_c), ("messidor", df_mss_c)]

# ── Plot 1 : class distribution ─────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
for ax, (name, df_) in zip(axes[:5], datasets_eda):
    if len(df_):
        counts = df_["label"].astype(int).value_counts().sort_index() \
                 .reindex(range(5), fill_value=0)
        bars = ax.bar(range(5), counts.values, color=GRADE_COLORS)
        ax.set_title(f"{name} — {len(df_):,} images", fontweight="bold")
        ax.set_xticks(range(5))
        ax.set_xticklabels([g.split("-")[0] for g in GRADE_NAMES])
        ax.set_xlabel("DR grade")
        ax.set_ylabel("count")
        for b, v in zip(bars, counts.values):
            ax.text(b.get_x() + b.get_width() / 2, v, f"{v:,}",
                    ha="center", va="bottom", fontsize=8)
    else:
        ax.set_title(f"{name} — empty")
        ax.axis("off")

combined = pd.concat([df_ for _, df_ in datasets_eda if len(df_)],
                     ignore_index=True)
if len(combined):
    counts = combined["label"].astype(int).value_counts().sort_index() \
             .reindex(range(5), fill_value=0)
    bars = axes[5].bar(range(5), counts.values, color=GRADE_COLORS)
    axes[5].set_title(f"COMBINED — {len(combined):,} images", fontweight="bold")
    axes[5].set_xticks(range(5))
    axes[5].set_xticklabels([g.split("-")[0] for g in GRADE_NAMES])
    axes[5].set_xlabel("DR grade")
    axes[5].set_ylabel("count")
    for b, v in zip(bars, counts.values):
        axes[5].text(b.get_x() + b.get_width() / 2, v, f"{v:,}",
                     ha="center", va="bottom", fontsize=8)

plt.suptitle("Class distribution across datasets (after cleaning)",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(LOG_DIR / "eda_class_distribution.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"  💾 saved → {LOG_DIR}/eda_class_distribution.png")


# ── Plot 2 : blur by grade ──────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
for ax, (name, df_) in zip(axes[:5], datasets_eda):
    if len(df_) and "laplacian" in df_.columns:
        data_per_grade = [df_[df_["label"].astype(int) == g]["laplacian"].values
                          for g in range(5)]
        bp = ax.boxplot(data_per_grade,
                        labels=[g.split("-")[0] for g in GRADE_NAMES],
                        patch_artist=True, showfliers=False)
        for patch, color in zip(bp["boxes"], GRADE_COLORS):
            patch.set_facecolor(color)
        ax.set_title(name, fontweight="bold")
        ax.set_xlabel("DR grade")
        ax.set_ylabel("Laplacian variance")
        ax.set_yscale("log")
    else:
        ax.set_title(f"{name} — no data")
        ax.axis("off")

if len(combined) and "laplacian" in combined.columns:
    data_per_grade = [combined[combined["label"].astype(int) == g]["laplacian"].values
                      for g in range(5)]
    bp = axes[5].boxplot(data_per_grade,
                         labels=[g.split("-")[0] for g in GRADE_NAMES],
                         patch_artist=True, showfliers=False)
    for patch, color in zip(bp["boxes"], GRADE_COLORS):
        patch.set_facecolor(color)
    axes[5].set_title("COMBINED", fontweight="bold")
    axes[5].set_xlabel("DR grade")
    axes[5].set_ylabel("Laplacian variance")
    axes[5].set_yscale("log")

plt.suptitle("Blur score by DR grade (higher = sharper)",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(LOG_DIR / "eda_blur_by_grade.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"  💾 saved → {LOG_DIR}/eda_blur_by_grade.png")


# ── Plot 3 : QC acceptance rate per dataset ─────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
names, accepts, rejects = [], [], []
for name, df_raw, df_c in [("aptos",    df_ap,    df_ap_c),
                            ("eyepacs",  df_ep,    df_ep_c),
                            ("ddr",      df_ddr,   df_ddr_c),
                            ("idrid",    df_idrid, df_idrid_c),
                            ("messidor", df_mss,   df_mss_c)]:
    names.append(name)
    accepts.append(len(df_c))
    rejects.append(len(df_raw) - len(df_c))

x = np.arange(len(names))
ax.bar(x, accepts, label="accepted", color="#2ecc71")
ax.bar(x, rejects, bottom=accepts, label="rejected", color="#e74c3c")
ax.set_xticks(x)
ax.set_xticklabels(names)
ax.set_ylabel("images")
ax.set_title("Blur + dedup QC: retained vs dropped per dataset")
ax.legend()
for i, (a, r) in enumerate(zip(accepts, rejects)):
    total = a + r
    if total:
        ax.text(i, a + r + max(accepts) * 0.01, f"{a / total * 100:.0f}% kept",
                ha="center", fontsize=9)
plt.tight_layout()
plt.savefig(LOG_DIR / "eda_qc_rates.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"  💾 saved → {LOG_DIR}/eda_qc_rates.png")


### 3.3 · Fundus validator — MobileNetV3-Small (for Streamlit inference gate)

Trains a lightweight binary classifier to distinguish genuine fundus photographs from any other image content.

**Purpose:** This model is **not** used to filter training data. It is saved to `EXPORT_DIR/fundus_validator.pt` and loaded exclusively by the Streamlit deployment app (§14) to reject non-fundus user uploads *before* grading is attempted.

**Design:**
- Architecture: MobileNetV3-Small with ImageNet-21K → ImageNet-1K pretrained weights (`mobilenetv3_small_100.miil_in21k_ft_in1k`, fallback to IN1K)
- Positives: sampled from the five **cleaned** datasets (`_c` parquets from §3.1)
- Negatives: procedurally generated synthetic images (noise, gradients, shapes, grayscale)
- Loss: `BCEWithLogitsLoss`  |  Threshold: sigmoid > 0.5 → fundus

Set `RETRAIN_VALIDATOR = True` to force rebuild even if checkpoint exists.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# STEP 3.3 — TRAIN MobileNetV3-Small FUNDUS VALIDATOR (Streamlit inference gate)
# -----------------------------------------------------------------------
# FIXED: cv2 drawing calls removed (deadlock on Windows), images pre-loaded
# into RAM before training to avoid per-batch disk I/O freeze.
# ═══════════════════════════════════════════════════════════════════════
from torch.utils.data import Dataset as _ValDS, DataLoader as _ValDL

VALIDATOR_PATH   = EXPORT_DIR / "fundus_validator.pt"
VALIDATOR_SIZE   = 224
VALIDATOR_EPOCHS = 3
VALIDATOR_BS     = 32
N_POS            = 500   # reduced from 4000 — enough for a gate classifier
N_NEG            = 500   # reduced from 4000

_IN21K_ARCH    = "mobilenetv3_small_100.miil_in21k_ft_in1k"
_FALLBACK_ARCH = "mobilenetv3_small_100"


def _create_validator_backbone() -> tuple:
    for arch in (_IN21K_ARCH, _FALLBACK_ARCH):
        try:
            m = timm.create_model(arch, pretrained=True, num_classes=1)
            print(f"  Loaded validator backbone : {arch}")
            return m, arch
        except Exception as exc:
            print(f"  ⚠️  {arch} unavailable ({exc.__class__.__name__}), trying fallback …")
    raise RuntimeError("No valid MobileNetV3-Small checkpoint could be loaded.")


if VALIDATOR_PATH.exists() and not globals().get("RETRAIN_VALIDATOR", False):
    print(f"  ✅ Fundus validator cached → {VALIDATOR_PATH}")
    print(f"     Set RETRAIN_VALIDATOR = True to rebuild.")
else:
    t0 = step_header("3.4", "TRAIN MobileNetV3-Small FUNDUS VALIDATOR")

    # ── Positive pool ──────────────────────────────────────────────────
    pos_pool = pd.concat(
        [df for df in (df_ap_c, df_ep_c, df_ddr_c, df_idrid_c, df_mss_c) if len(df)],
        ignore_index=True,
    )
    pos_pool = pos_pool.sample(
        n=min(N_POS, len(pos_pool)), random_state=42
    ).reset_index(drop=True)
    print(f"  Positive samples : {len(pos_pool):,}")

    # ── Negative generator — pure numpy only, NO cv2 drawing ──────────
    def _gen_negative(idx: int, size: int = VALIDATOR_SIZE, seed_base: int = 12345) -> np.ndarray:
        rng = np.random.RandomState(seed_base + idx)
        kind = rng.randint(0, 3)
        if kind == 0:   # uniform noise
            return (rng.rand(size, size, 3) * 255).astype(np.uint8)
        elif kind == 1: # gradient field
            x = np.linspace(0, 255, size).astype(np.uint8)
            img = np.tile(x, (size, 1))[:, :, None].repeat(3, axis=2)
            img = np.clip(
                img.astype(np.int16) + (rng.rand(size, size, 3) * 40 - 20), 0, 255
            ).astype(np.uint8)
            for c in range(3):
                img[..., c] = (img[..., c] * rng.uniform(0.3, 1.0)).astype(np.uint8)
            return img
        else:           # pure grayscale
            g = rng.randint(0, 256, (size, size), dtype=np.uint8)
            return np.stack([g, g, g], axis=-1)

    def _load_pos(path: str, size: int = VALIDATOR_SIZE) -> np.ndarray:
        img = cv2.imread(path)
        if img is None:
            return _gen_negative(0, size)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        s = size / max(h, w)
        img = cv2.resize(img, (max(1, int(w * s)), max(1, int(h * s))),
                         interpolation=cv2.INTER_AREA)
        canvas = np.zeros((size, size, 3), dtype=np.uint8)
        y0 = (size - img.shape[0]) // 2
        x0 = (size - img.shape[1]) // 2
        canvas[y0:y0 + img.shape[0], x0:x0 + img.shape[1]] = img
        return canvas

    # ── Pre-load ALL images into RAM before training ───────────────────
    print("  📦 Pre-loading positive images into RAM …")
    pos_imgs_all = []
    for p in tqdm(pos_pool["img_path"].tolist(), desc="loading positives", leave=False):
        pos_imgs_all.append(_load_pos(p))
    print(f"  ✅ {len(pos_imgs_all):,} positives loaded")

    print("  📦 Pre-generating negatives …")
    neg_imgs_all = [_gen_negative(i) for i in range(N_NEG)]
    print(f"  ✅ {len(neg_imgs_all):,} negatives generated")

    # ── Dataset class (reads from RAM lists, no disk I/O during training)
    class _ValidatorDataset(_ValDS):
        _MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
        _STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

        def __init__(self, imgs, labels, train=True):
            self.imgs   = imgs
            self.labels = labels
            self.train  = train

        def __len__(self):
            return len(self.imgs)

        def __getitem__(self, idx):
            img   = self.imgs[idx].copy()
            label = float(self.labels[idx])
            if self.train and np.random.rand() < 0.5:
                img = np.ascontiguousarray(img[:, ::-1])
            if self.train and np.random.rand() < 0.5:
                img = np.ascontiguousarray(img[::-1, :])
            img = (img.astype(np.float32) / 255.0 - self._MEAN) / self._STD
            return (torch.from_numpy(img.transpose(2, 0, 1)).float(),
                    torch.tensor(label, dtype=torch.float32))

    # ── Split into train / val ─────────────────────────────────────────
    n_tr_pos = int(len(pos_imgs_all) * 0.9)
    n_tr_neg = int(len(neg_imgs_all) * 0.9)

    tr_imgs   = pos_imgs_all[:n_tr_pos]   + neg_imgs_all[:n_tr_neg]
    tr_labels = [1] * n_tr_pos             + [0] * n_tr_neg
    va_imgs   = pos_imgs_all[n_tr_pos:]   + neg_imgs_all[n_tr_neg:]
    va_labels = [1] * len(pos_imgs_all[n_tr_pos:]) + [0] * len(neg_imgs_all[n_tr_neg:])

    tr_ds = _ValidatorDataset(tr_imgs, tr_labels, train=True)
    va_ds = _ValidatorDataset(va_imgs, va_labels, train=False)
    tr_dl = _ValDL(tr_ds, batch_size=VALIDATOR_BS, shuffle=True,
                   num_workers=0, pin_memory=False)
    va_dl = _ValDL(va_ds, batch_size=VALIDATOR_BS, shuffle=False,
                   num_workers=0, pin_memory=False)
    print(f"  Train split : {len(tr_ds):,} samples  |  Val split : {len(va_ds):,} samples")

    # ── Model ──────────────────────────────────────────────────────────
    validator, _arch_used = _create_validator_backbone()
    validator = validator.to(DEVICE)
    opt    = torch.optim.AdamW(validator.parameters(), lr=3e-4, weight_decay=1e-4)
    crit   = nn.BCEWithLogitsLoss()
    scaler = torch.cuda.amp.GradScaler(enabled=AMP_ENABLED)

    # ── Training loop ──────────────────────────────────────────────────
    best_acc = 0.0
    for epoch in range(VALIDATOR_EPOCHS):
        validator.train()
        tr_loss = tr_n = 0
        for x, y in tqdm(tr_dl, desc=f"val-epoch {epoch + 1}/{VALIDATOR_EPOCHS}", leave=False):
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=AMP_ENABLED):
                loss = crit(validator(x).squeeze(-1), y)
            opt.zero_grad(set_to_none=True)
            if AMP_ENABLED:
                scaler.scale(loss).backward()
                scaler.step(opt)
                scaler.update()
            else:
                loss.backward()
                opt.step()
            tr_loss += float(loss.item()) * x.size(0)
            tr_n    += x.size(0)
        tr_loss /= max(1, tr_n)

        validator.eval()
        correct = total = 0
        with torch.no_grad():
            for x, y in va_dl:
                x, y = x.to(DEVICE), y.to(DEVICE)
                with torch.cuda.amp.autocast(enabled=AMP_ENABLED):
                    preds = (torch.sigmoid(validator(x).squeeze(-1)) > 0.5).float()
                correct += (preds == y).sum().item()
                total   += y.size(0)
        val_acc = correct / max(1, total)
        print(f"  epoch {epoch + 1}/{VALIDATOR_EPOCHS} | train_loss={tr_loss:.4f} | val_acc={val_acc:.4f}")

        if val_acc > best_acc:
            best_acc = val_acc
            torch.save({
                "state_dict": validator.state_dict(),
                "arch":       _arch_used,
                "size":       VALIDATOR_SIZE,
                "val_acc":    val_acc,
            }, VALIDATOR_PATH)
            print(f"    💾 saved → {VALIDATOR_PATH.name}  (acc={val_acc:.4f})")

    step_done(t0)
    print(f"\n  ✅ Best validation accuracy : {best_acc:.4f}")
    print(f"  ✅ Architecture used        : {_arch_used}")
    print(f"  ✅ Checkpoint saved to      : {VALIDATOR_PATH}")

## 4 · Final training-dataset extraction — *after cleaning*

Now that every dataset has passed blur+dedup QC (§3.1), we assemble the training set the model will actually see. This step is deliberately placed **after** Section 3 so the final pool contains zero blurry or duplicate images.

Two artefacts are produced:

| Artefact | Contents | Used by |
|---|---|---|
| `stage1_pool.parquet` | APTOS (with fold 0-4) + EyePACS + DDR (fold = -1, always train) | Stage 1 K-fold trainer |
| `split_{train,val,test}.parquet` | Stratified 80/10/10 of the combined clean pool | Ad-hoc single-model training / sanity checks |

### 4.1 · Build Stage-1 pool with K-fold split on APTOS

APTOS is split into 5 stratified folds → rotating validation. EyePACS + DDR get `fold = -1` → always in training. This mimics the APTOS competition rules: validation must come from the APTOS distribution only.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 4.1 — BUILD STAGE-1 POOL + 5-FOLD SPLIT ON APTOS
# ═══════════════════════════════════════════════════════════════
t0 = step_header("4.1", "STAGE-1 POOL + K-FOLD SPLIT")

ap_clean    = df_ap_c.copy()    if len(df_ap_c)    else df_ap_c
ep_clean    = df_ep_c.copy()    if len(df_ep_c)    else df_ep_c
ddr_clean   = df_ddr_c.copy()   if len(df_ddr_c)   else df_ddr_c
idrid_clean = df_idrid_c.copy() if len(df_idrid_c) else df_idrid_c
mss_clean   = df_mss_c.copy()   if len(df_mss_c)   else df_mss_c

# APTOS → stratified K-fold
ap_clean = ap_clean.reset_index(drop=True)
ap_clean["fold"] = -1
if len(ap_clean):
    skf = StratifiedKFold(n_splits=NUM_FOLDS, shuffle=True, random_state=42)
    for fi, (_, va_idx) in enumerate(
            skf.split(ap_clean, ap_clean["label"].astype(int))):
        ap_clean.loc[va_idx, "fold"] = fi

# Auxiliary datasets → always train (fold = -1)
for df_ in (ep_clean, ddr_clean):
    if len(df_):
        df_["fold"] = -1

df_stage1 = pd.concat([ap_clean, ep_clean, ddr_clean], ignore_index=True)
df_stage1["label"] = df_stage1["label"].astype(int)
df_stage1.to_parquet(INDEX_DIR / "stage1_pool.parquet", index=False)

# Save final clean indices used later by Stages 2 + 3
idrid_clean.to_parquet(INDEX_DIR / "idrid_final.parquet",    index=False)
mss_clean.to_parquet(  INDEX_DIR / "messidor_final.parquet", index=False)

print(f"  Stage-1 pool:       {len(df_stage1):,} images")
print(f"    APTOS (K-fold):   {len(ap_clean):,}")
print(f"    EyePACS (train):  {len(ep_clean):,}")
print(f"    DDR (train):      {len(ddr_clean):,}")
print()
if len(ap_clean):
    print("  APTOS per-fold class balance:")
    print(ap_clean.groupby(["fold", "label"])
                  .size().unstack(fill_value=0).to_string())
    print("\n  APTOS fold sizes:",
          ap_clean["fold"].value_counts().sort_index().to_dict())

step_done(t0)


### 4.2 · Stratified 80/10/10 split (for quick experiments)

Optional sibling of the K-fold split — used by ad-hoc notebooks that want a simple train/val/test without rotating folds.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 4.2 — STRATIFIED 80/10/10 TRAIN/VAL/TEST SPLIT
# ═══════════════════════════════════════════════════════════════
t0 = step_header("4.2", "STRATIFIED 80/10/10 SPLIT")

pool = pd.concat([df_ap_c, df_ep_c, df_ddr_c], ignore_index=True)
pool["label"] = pool["label"].astype(int)
pool = pool.reset_index(drop=True)
print(f"  Combined pool: {len(pool):,} images")

tr_df, tmp_df = train_test_split(
    pool, test_size=0.20, random_state=42, stratify=pool["label"])
va_df, te_df = train_test_split(
    tmp_df, test_size=0.50, random_state=42, stratify=tmp_df["label"])

tr_df = tr_df.reset_index(drop=True)
va_df = va_df.reset_index(drop=True)
te_df = te_df.reset_index(drop=True)

tr_df.to_parquet(INDEX_DIR / "split_train.parquet", index=False)
va_df.to_parquet(INDEX_DIR / "split_val.parquet",   index=False)
te_df.to_parquet(INDEX_DIR / "split_test.parquet",  index=False)


def _show_dist(df_, label):
    dist = df_["label"].value_counts().sort_index().to_dict()
    print(f"    {label:6s} ({len(df_):6,}): {dist}")


print("  Split sizes + class balance:")
_show_dist(tr_df, "train")
_show_dist(va_df, "val")
_show_dist(te_df, "test")
step_done(t0)


## 5 · Modeling primitives

Re-usable building blocks: augmentations, `Dataset`, `DataLoader`, model architecture, EMA, metrics.

All of these are pure-definition cells — nothing trains here, just pure functions & classes that the training cells will import.

### 5.1 · Augmentations, `Dataset`, sampler, loader

Train-time augs lean into the fact that fundus images are rotationally invariant (180° rotation, both flips) and simulate variable capture conditions (brightness/contrast jitter, coarse dropout).

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 5.1 — AUGMENTATIONS + DATASET + DATALOADER
# ═══════════════════════════════════════════════════════════════
MEAN = (0.485, 0.456, 0.406)
STD  = (0.229, 0.224, 0.225)


def build_train_tf(size: int):
    return A.Compose([
        A.LongestMaxSize(max_size=size),
        A.PadIfNeeded(min_height=size, min_width=size,
                      border_mode=cv2.BORDER_CONSTANT, value=0),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.10,
                           rotate_limit=180,
                           border_mode=cv2.BORDER_CONSTANT, value=0, p=0.7),
        A.RandomBrightnessContrast(brightness_limit=0.15,
                                   contrast_limit=0.15, p=0.5),
        A.CoarseDropout(max_holes=6,
                        max_height=int(size * 0.1),
                        max_width=int(size * 0.1), p=0.3),
        A.Normalize(mean=MEAN, std=STD),
        ToTensorV2(),
    ])


def build_val_tf(size: int):
    return A.Compose([
        A.LongestMaxSize(max_size=size),
        A.PadIfNeeded(min_height=size, min_width=size,
                      border_mode=cv2.BORDER_CONSTANT, value=0),
        A.Normalize(mean=MEAN, std=STD),
        ToTensorV2(),
    ])


class DRDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform=None, has_label=True):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.has_label = has_label

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = cv2.imread(row["img_path"])
        if img is None:
            img = np.zeros((256, 256, 3), dtype=np.uint8)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if self.transform is not None:
            img = self.transform(image=img)["image"]
        if self.has_label:
            return img, float(row["label"])
        return img


def build_sampler(df: pd.DataFrame) -> WeightedRandomSampler:
    """Inverse-frequency sampler — addresses severe class imbalance."""
    counts = df["label"].value_counts().to_dict()
    inv = {k: 1.0 / v for k, v in counts.items()}
    w = df["label"].map(inv).values.astype("float64")
    return WeightedRandomSampler(w, num_samples=len(w), replacement=True)


def make_loader(ds, batch, shuffle=False, sampler=None, drop_last=False):
    return DataLoader(
        ds, batch_size=batch,
        shuffle=(sampler is None and shuffle),
        sampler=sampler,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=drop_last,
        persistent_workers=(NUM_WORKERS > 0),
    )


print("✅ Transforms / Dataset / Sampler / Loader helpers ready.")


### 5.2 · Model — GeM pooling + regression head

- **GeM pooling** (arXiv:1711.02512) — learnable-`p` generalisation of avg/max pooling; consistently 0.5–1 QWK points better than plain GAP on fundus.
- **Regression head** — single scalar output; we fit `SmoothL1Loss` against the grade and optimise thresholds post-hoc for QWK.
- **EMA** — decay 0.9999 tracks a smoothed weight copy; used at validation and for the final checkpoint.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 5.2 — GeM POOLING + DR MODEL + EMA
# ═══════════════════════════════════════════════════════════════
class GeM(nn.Module):
    """Generalized Mean Pooling (learnable p)."""

    def __init__(self, p=3.0, eps=1e-6):
        super().__init__()
        self.p = Parameter(torch.ones(1) * p)
        self.eps = eps

    def forward(self, x):
        return F.avg_pool2d(x.clamp(min=self.eps).pow(self.p),
                            (x.size(-2), x.size(-1))).pow(1.0 / self.p)

    def __repr__(self):
        return f"GeM(p={float(self.p):.3f})"


class DRModel(nn.Module):
    def __init__(self, backbone_name: str, pretrained=True, drop=0.5):
        super().__init__()
        self.backbone = timm.create_model(
            backbone_name, pretrained=pretrained,
            num_classes=0, global_pool="",
        )
        feat = self.backbone.num_features
        self.pool = GeM()
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(feat, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(drop),
            nn.Linear(256, 1),
        )

    def forward(self, x):
        f = self.backbone(x)
        if f.dim() == 2:                        # some backbones already pool
            p = f.unsqueeze(-1).unsqueeze(-1)
        else:
            p = self.pool(f)
        return self.head(p).squeeze(-1)

    def freeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad_(False)

    def unfreeze_all(self):
        for p in self.parameters():
            p.requires_grad_(True)


class EMA:
    """Exponential Moving Average of model weights."""

    def __init__(self, model, decay=0.9999):
        self.decay = decay
        self.shadow = {n: p.data.clone().detach()
                       for n, p in model.named_parameters() if p.requires_grad}
        self.backup = {}

    def update(self, model):
        for n, p in model.named_parameters():
            if p.requires_grad and n in self.shadow:
                self.shadow[n].mul_(self.decay).add_(p.data, alpha=1 - self.decay)

    def apply(self, model):
        self.backup = {}
        for n, p in model.named_parameters():
            if n in self.shadow:
                self.backup[n] = p.data.clone()
                p.data.copy_(self.shadow[n])

    def restore(self, model):
        for n, p in model.named_parameters():
            if n in self.backup:
                p.data.copy_(self.backup[n])
        self.backup = {}

    def state_dict(self):
        return {"shadow": self.shadow, "decay": self.decay}

    def load_state_dict(self, st):
        self.shadow = st["shadow"]
        self.decay = st.get("decay", self.decay)


def build_model(backbone_name: str, pretrained=True) -> DRModel:
    return DRModel(backbone_name, pretrained=pretrained, drop=0.5)


print("✅ Model / EMA classes ready.")
for m in MODEL_CONFIGS:
    print(f"   • {m['name']}: {m['backbone']}")


### 5.3 · Metrics + threshold optimisation

QWK (Quadratic Weighted Kappa) is the APTOS competition metric. For regression outputs we use 4 real-valued cut-points `(t0, t1, t2, t3)` and optimise them on OOF predictions via coordinate descent.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 5.3 — METRICS + THRESHOLD OPTIMIZATION
# ═══════════════════════════════════════════════════════════════
def preds_to_labels(preds, thresholds):
    """Continuous preds → integer labels via sorted thresholds."""
    preds = np.asarray(preds)
    labels = np.zeros_like(preds, dtype=np.int64)
    for i, t in enumerate(thresholds):
        labels[preds > t] = i + 1
    return labels


def qwk(y_true, y_pred):
    return cohen_kappa_score(y_true, y_pred, weights="quadratic")


def optimize_thresholds(oof_preds, oof_true,
                        init=(0.5, 1.5, 2.5, 3.5),
                        n_iters=4, grid_step=0.01):
    """Coordinate-descent search on QWK."""
    thr = list(init)
    best = qwk(oof_true, preds_to_labels(oof_preds, thr))
    for _ in range(n_iters):
        for i in range(len(thr)):
            lo = (thr[i - 1] if i > 0 else 0.0) + 1e-3
            hi = (thr[i + 1] if i < len(thr) - 1 else 4.0) - 1e-3
            grid = np.arange(lo, hi, grid_step)
            qs = []
            for g in grid:
                cand = thr.copy()
                cand[i] = g
                qs.append(qwk(oof_true, preds_to_labels(oof_preds, cand)))
            if not qs:
                continue
            j = int(np.argmax(qs))
            if qs[j] >= best:
                best = qs[j]
                thr[i] = grid[j]
    return thr, best


def eval_regression(preds, y_true, thresholds=None):
    """Full metrics dict. Optimise thresholds if none supplied."""
    preds = np.asarray(preds)
    y_true = np.asarray(y_true, dtype=int)
    if thresholds is None:
        thresholds, _ = optimize_thresholds(preds, y_true)
    labels = preds_to_labels(preds, thresholds)
    return {
        "qwk":        qwk(y_true, labels),
        "accuracy":   accuracy_score(y_true, labels),
        "f1_macro":   f1_score(y_true, labels, average="macro", zero_division=0),
        "precision":  precision_score(y_true, labels, average="macro", zero_division=0),
        "recall":     recall_score(y_true, labels, average="macro", zero_division=0),
        "thresholds": list(map(float, thresholds)),
        "labels":     labels,
    }


print("✅ Metric helpers ready.")


## 6 · Dataloader sanity check

Before launching 30 runs of Stage-1 training, pull a single batch from the real loader and eyeball it. If the images look like garbage here, they'll look like garbage in the model too.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# DATALOADER SANITY CHECK
# ═══════════════════════════════════════════════════════════════
t0 = step_header(6, "DATALOADER SANITY CHECK")

if len(df_stage1) == 0:
    print("  ⚠️  df_stage1 is empty — check Sections 2–4 before training.")
else:
    _tmp_ds = DRDataset(df_stage1.head(64), transform=build_train_tf(256))
    _tmp_dl = make_loader(_tmp_ds, batch=8)

    xb, yb = next(iter(_tmp_dl))
    print(f"  Batch x: shape={tuple(xb.shape)}  dtype={xb.dtype}  "
          f"range=[{float(xb.min()):.3f}, {float(xb.max()):.3f}]")
    print(f"  Batch y: shape={tuple(yb.shape)}  values={[int(v) for v in yb.tolist()]}")

    n_show = min(8, len(xb))
    fig, ax = plt.subplots(1, n_show, figsize=(16, 3))
    if n_show == 1:
        ax = [ax]
    for i, a in enumerate(ax):
        im = xb[i].permute(1, 2, 0).numpy()
        im = im * np.array(STD) + np.array(MEAN)
        im = np.clip(im, 0, 1)
        a.imshow(im)
        a.axis("off")
        a.set_title(f"y={int(yb[i])}", fontsize=8)
    plt.tight_layout()
    plt.show()

step_done(t0)


## 7 · Stage 1 training — K-fold cross-validation

The production trainer. For every `(backbone, seed, fold)` combination:

1. **Phase 1 (P1-Freeze)** — backbone frozen, train the head to warm-start.
2. **Phase 2 (P2-Unfreeze)** — full fine-tune with discriminative LRs (backbone at LR/10, head at full LR).
3. **Phase 3 (P3-Polish)** — a few low-LR epochs to settle.

Every single epoch writes an atomic checkpoint, so a crash mid-run just resumes from the last completed epoch. EMA weights are used for validation and for the saved `*_best.pt`.

### 7.1 · Trainer

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 7.1 — STAGE 1 TRAINER (resumable, fold-granular)
# ═══════════════════════════════════════════════════════════════
def build_optim(model, lr, wd):
    """Discriminative LR: backbone @ lr/10, head @ lr."""
    back = [p for p in model.backbone.parameters() if p.requires_grad]
    head = list(model.pool.parameters()) + list(model.head.parameters())
    groups = []
    if back:
        groups.append({"params": back, "lr": lr / 10})
    if head:
        groups.append({"params": head, "lr": lr})
    return torch.optim.AdamW(groups, weight_decay=wd)


def cosine_lr(step, total, warmup):
    if step < warmup:
        return (step + 1) / max(1, warmup)
    return 0.5 * (1 + math.cos(math.pi * (step - warmup) / max(1, total - warmup)))


def train_one_epoch(model, loader, optimizer, scaler, criterion, ema, accum_steps=1):
    model.train()
    total, n = 0.0, 0
    optimizer.zero_grad(set_to_none=True)
    for step, (x, y) in enumerate(tqdm(loader, desc="train", leave=False)):
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True).float()
        with torch.cuda.amp.autocast(enabled=AMP_ENABLED):
            pred = model(x)
            loss = criterion(pred, y) / accum_steps
        if AMP_ENABLED:
            scaler.scale(loss).backward()
        else:
            loss.backward()
        if (step + 1) % accum_steps == 0:
            if AMP_ENABLED:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                scaler.step(optimizer)
                scaler.update()
            else:
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            if ema is not None:
                ema.update(model)
        total += float(loss.item()) * accum_steps * x.size(0)
        n += x.size(0)
    return total / max(1, n)


@torch.no_grad()
def predict(model, loader):
    model.eval()
    preds, ys = [], []
    for batch in tqdm(loader, desc="predict", leave=False):
        if isinstance(batch, (list, tuple)) and len(batch) == 2:
            x, y = batch
            ys.append(y.numpy())
        else:
            x = batch
        x = x.to(DEVICE, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=AMP_ENABLED):
            preds.append(model(x).float().cpu().numpy())
    preds = np.concatenate(preds)
    return (preds, np.concatenate(ys)) if ys else (preds, None)


def train_one_fold(backbone_name, model_name, seed_val, fold, df_pool):
    """Train a single (backbone, seed, fold). Returns (best_qwk, oof_preds, oof_idx)."""
    tag = f"{model_name}_s{seed_val}_f{fold}"
    best_ckpt = CKPT_DIR / f"{tag}_best.pt"
    oof_npy   = OOF_DIR  / f"{tag}_oof.npy"
    oof_idx   = OOF_DIR  / f"{tag}_idx.npy"

    if is_done(tag) and best_ckpt.exists() and not FORCE_STAGE1_RETRAIN:
        st = safe_load(best_ckpt, "cpu")
        print(f"    ✅ {tag}: QWK={st.get('val_qwk', 0):.4f}  (cached)")
        oof = np.load(oof_npy) if oof_npy.exists() else None
        idx = np.load(oof_idx) if oof_idx.exists() else None
        return st.get("val_qwk", 0), oof, idx

    seed_everything(seed_val + fold)

    val_mask = (df_pool["dataset"] == "aptos") & (df_pool["fold"] == fold)
    val_orig_idx = df_pool.index[val_mask].to_numpy()
    df_val   = df_pool[val_mask].reset_index(drop=True)
    df_train = df_pool[~val_mask].reset_index(drop=True)

    print(f"\n  ▶ {tag}   (train={len(df_train):,}  val={len(df_val):,})")

    model = build_model(backbone_name, pretrained=True).to(DEVICE)
    ema = EMA(model, decay=EMA_DECAY)
    criterion = nn.SmoothL1Loss()

    best_qwk = -1.0
    best_thr = None
    oof_final = None

    cfg_entry = next(m for m in MODEL_CONFIGS if m["backbone"] == backbone_name)
    phase_sizes = (cfg_entry["sz_p1"], cfg_entry["sz_p2"], cfg_entry["sz_p2"])

    for pi, phase in enumerate(PHASES):
        if phase["epochs"] <= 0:
            continue
        size  = phase_sizes[pi]
        bs    = phase["batch"]
        accum = phase["accum"]
        lr    = phase["lr"]
        nep   = phase["epochs"]
        pname = phase["name"]
        p_tag  = f"{tag}_p{pi}"
        p_ckpt = CKPT_DIR / f"{p_tag}.pt"

        if phase["freeze"]:
            model.freeze_backbone()
        else:
            model.unfreeze_all()
        n_train_param = sum(p.numel() for p in model.parameters() if p.requires_grad)

        tr_ds = DRDataset(df_train, transform=build_train_tf(size))
        va_ds = DRDataset(df_val,   transform=build_val_tf(size))
        tr_dl = make_loader(tr_ds, bs,
                            sampler=build_sampler(df_train), drop_last=True)
        va_dl = make_loader(va_ds, bs, shuffle=False)

        optimizer = build_optim(model, lr, WD)
        warmup = min(2, nep // 5)
        scheduler = torch.optim.lr_scheduler.LambdaLR(
            optimizer, lambda e: cosine_lr(e, nep, warmup))
        scaler = torch.cuda.amp.GradScaler(enabled=AMP_ENABLED)

        # Resume within phase
        ep_start = 0
        if p_ckpt.exists():
            try:
                st = safe_load(p_ckpt, DEVICE)
                model.load_state_dict(st["model"])
                optimizer.load_state_dict(st["opt"])
                scheduler.load_state_dict(st["sched"])
                if st.get("ema") is not None:
                    ema.load_state_dict(st["ema"])
                if st.get("scaler") is not None and AMP_ENABLED:
                    scaler.load_state_dict(st["scaler"])
                ep_start = st.get("epoch", 0) + 1
                print(f"    [{pname}] resume from epoch {ep_start}/{nep}")
            except Exception as e:
                print(f"    [{pname}] resume failed ({e}); starting fresh")

        print(f"    [{pname}] size={size}px  ep {ep_start + 1}..{nep}  "
              f"bs={bs}x{accum}  params={n_train_param / 1e6:.1f}M  lr={lr:g}")

        for ep in range(ep_start, nep):
            t_ep = time.time()
            train_loss = train_one_epoch(model, tr_dl, optimizer, scaler,
                                         criterion, ema, accum_steps=accum)
            scheduler.step()

            # Validate with EMA weights
            ema.apply(model)
            val_preds, val_true = predict(model, va_dl)
            ema.restore(model)

            thr, _ = optimize_thresholds(val_preds, val_true.astype(int))
            cur_qwk = qwk(val_true.astype(int), preds_to_labels(val_preds, thr))
            cur_acc = accuracy_score(val_true.astype(int),
                                     preds_to_labels(val_preds, thr))
            print(f"      ep {ep + 1}/{nep}  loss={train_loss:.4f}  "
                  f"val_qwk={cur_qwk:.4f}  val_acc={cur_acc:.4f}  "
                  f"({time.time() - t_ep:.0f}s)")

            safe_save({
                "model":   model.state_dict(),
                "opt":     optimizer.state_dict(),
                "sched":   scheduler.state_dict(),
                "ema":     ema.state_dict(),
                "scaler":  scaler.state_dict() if AMP_ENABLED else None,
                "epoch":   ep,
                "val_qwk": cur_qwk,
            }, p_ckpt)

            if cur_qwk > best_qwk:
                best_qwk = cur_qwk
                best_thr = thr
                ema.apply(model)
                safe_save({
                    "model":      model.state_dict(),
                    "val_qwk":    cur_qwk,
                    "thresholds": thr,
                    "backbone":   backbone_name,
                    "size":       size,
                }, best_ckpt)
                ema.restore(model)
                oof_final = val_preds

            gc.collect()
            if DEVICE_KIND == "cuda":
                torch.cuda.empty_cache()

    if oof_final is not None:
        np.save(oof_npy, oof_final)
        np.save(oof_idx, val_orig_idx)
        np.save(OOF_DIR / f"{tag}_paths.npy",
                df_val["img_path"].values.astype(object))
    mark_done(tag)
    print(f"    ✓ {tag} done — best QWK={best_qwk:.4f}")
    return best_qwk, oof_final, df_val.index.to_numpy()


print("✅ Stage-1 trainer ready.")


### 7.2 · Run Stage 1

At full scale: `3 backbones × 2 seeds × 5 folds = 30 runs`. On an RTX 2050 this is ~20-30 hours depending on backbone. All 30 slots are resumable — a kernel crash means nothing is lost.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 7.2 — RUN STAGE 1  (15-day reduced: 1 backbone × 1 seed × 3 folds = 3 fold-runs)
# ═══════════════════════════════════════════════════════════════
t0 = step_header("7.2", "STAGE 1 TRAINING")
print(f"  Pipeline: {len(MODEL_CONFIGS)} backbone × {len(SEEDS)} seeds × {NUM_FOLDS} folds = {len(MODEL_CONFIGS)*len(SEEDS)*NUM_FOLDS} fold-runs")
print(f"  Phases per fold: {[(p['name'], p['epochs']) for p in PHASES]}  (total 14 epochs)")
print(f"  num_workers={NUM_WORKERS}  device={DEVICE}  AMP={AMP_ENABLED}")
print(f"  Expected wall-clock: ~6–10 days for all 3 folds (V2-L on RTX 2050 4GB)")
print("  Loading first batch (this can take 30–120s on cold start; first iteration triggers CUDA kernel compile)...", flush=True)

stage1_results = []
for cfg in MODEL_CONFIGS:
    for seed in SEEDS:
        for fold in range(NUM_FOLDS):
            print(f"\n  ─── Starting {cfg['name']} seed={seed} fold={fold} ───", flush=True)
            qwk_v, oof, idx = train_one_fold(
                cfg["backbone"], cfg["name"], seed, fold, df_stage1
            )
            stage1_results.append({
                "name": cfg["name"], "backbone": cfg["backbone"],
                "seed": seed, "fold": fold, "val_qwk": qwk_v,
            })

res_df = pd.DataFrame(stage1_results)
res_df.to_csv(LOG_DIR / "stage1_results.csv", index=False)
print("\n  Per-fold QWK summary:")
print(res_df.pivot_table(index=["name", "seed"], columns="fold",
                         values="val_qwk").round(4).to_string())

step_done(t0)


## 8 · OOF ensemble + global threshold optimisation

Each APTOS image appears in exactly one validation fold, so concatenating all 5 folds per (backbone, seed) yields a full OOF prediction for the whole APTOS dataset. Averaging across seeds, then across backbones, gives the final ensemble vector.

We then run `optimize_thresholds` once more on the full OOF ensemble → the thresholds that get shipped with the model.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 8 — OOF ENSEMBLE + GLOBAL THRESHOLDS
# ═══════════════════════════════════════════════════════════════
t0 = step_header(8, "OOF ENSEMBLE + THRESHOLDS")

df_aptos_only = df_stage1[df_stage1["dataset"] == "aptos"].reset_index(drop=True)
oof_matrix = np.full((len(df_aptos_only),
                      len(MODEL_CONFIGS) * len(SEEDS)), np.nan)

col = 0
col_names = []
for cfg in MODEL_CONFIGS:
    for seed in SEEDS:
        col_names.append(f"{cfg['name']}_s{seed}")
        for fold in range(NUM_FOLDS):
            tag = f"{cfg['name']}_s{seed}_f{fold}"
            oof_npy   = OOF_DIR / f"{tag}_oof.npy"
            paths_npy = OOF_DIR / f"{tag}_paths.npy"
            if oof_npy.exists() and paths_npy.exists():
                preds = np.load(oof_npy)
                paths = np.load(paths_npy, allow_pickle=True)
                positions = df_aptos_only.reset_index().set_index("img_path")["index"]
                for p, pr in zip(paths, preds):
                    pos = positions.get(str(p))
                    if pos is not None:
                        oof_matrix[pos, col] = pr
        col += 1

per_model = []
for i, _ in enumerate(MODEL_CONFIGS):
    cols = list(range(i * len(SEEDS), (i + 1) * len(SEEDS)))
    per_model.append(np.nanmean(oof_matrix[:, cols], axis=1))
per_model = np.stack(per_model, axis=1)
ensemble = np.nanmean(per_model, axis=1)

mask = ~np.isnan(ensemble)
y = df_aptos_only.loc[mask, "label"].astype(int).values
p = ensemble[mask]

thr, _ = optimize_thresholds(p, y)
metrics = eval_regression(p, y, thresholds=thr)

print(f"  Ensemble OOF — {len(p):,} APTOS images")
print(f"    QWK       : {metrics['qwk']:.4f}")
print(f"    Accuracy  : {metrics['accuracy']:.4f}")
print(f"    F1 (macro): {metrics['f1_macro']:.4f}")
print(f"    Precision : {metrics['precision']:.4f}")
print(f"    Recall    : {metrics['recall']:.4f}")
print(f"    Thresholds: {[round(t, 3) for t in metrics['thresholds']]}")

np.save(OOF_DIR / "ensemble_preds.npy", p)
np.save(OOF_DIR / "ensemble_true.npy",  y)
(EXPORT_DIR / "thresholds.json").write_text(json.dumps({
    "thresholds": metrics["thresholds"],
    "qwk":        metrics["qwk"],
    "n_samples":  int(len(p)),
}, indent=2))

cm = confusion_matrix(y, metrics["labels"], labels=list(range(NUM_CLASSES)))
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=range(NUM_CLASSES), yticklabels=range(NUM_CLASSES))
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title(f"OOF Confusion Matrix  (QWK={metrics['qwk']:.3f})")
plt.tight_layout()
plt.savefig(LOG_DIR / "oof_cm.png", dpi=120)
plt.show()

step_done(t0)


### 8.1 · Full metrics visualisation

Six-panel dashboard of the OOF ensemble: confusion-matrix counts + row-%, per-class P/R/F1, per-grade accuracy, prediction-distribution by true class, and cumulative-within-N-grades accuracy.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 8.1 — FULL METRICS VISUALISATION (6-panel dashboard)
# ═══════════════════════════════════════════════════════════════
from sklearn.metrics import precision_recall_fscore_support

p_path = OOF_DIR / "ensemble_preds.npy"
y_path = OOF_DIR / "ensemble_true.npy"
thr_path = EXPORT_DIR / "thresholds.json"

if not (p_path.exists() and y_path.exists() and thr_path.exists()):
    print("  ⚠  Run Section 8 first — ensemble arrays not yet persisted.")
else:
    p_oof = np.load(p_path)
    y_true = np.load(y_path)
    thresholds = np.array(json.loads(thr_path.read_text())["thresholds"])
    y_pred = preds_to_labels(p_oof, thresholds)

    fig, axes = plt.subplots(2, 3, figsize=(16, 10))

    cm = confusion_matrix(y_true, y_pred, labels=list(range(5)))
    im = axes[0, 0].imshow(cm, cmap="Blues")
    axes[0, 0].set_title("Confusion matrix (counts)", fontweight="bold")
    axes[0, 0].set_xticks(range(5))
    axes[0, 0].set_yticks(range(5))
    axes[0, 0].set_xlabel("predicted")
    axes[0, 0].set_ylabel("true")
    for i in range(5):
        for j in range(5):
            axes[0, 0].text(j, i, f"{cm[i, j]:,}", ha="center", va="center",
                            color="white" if cm[i, j] > cm.max() / 2 else "black",
                            fontsize=9)
    plt.colorbar(im, ax=axes[0, 0], fraction=0.046)

    cmn = cm / np.clip(cm.sum(axis=1, keepdims=True), 1, None) * 100
    axes[0, 1].imshow(cmn, cmap="Blues", vmin=0, vmax=100)
    axes[0, 1].set_title("Confusion matrix (row %)", fontweight="bold")
    axes[0, 1].set_xticks(range(5))
    axes[0, 1].set_yticks(range(5))
    axes[0, 1].set_xlabel("predicted")
    axes[0, 1].set_ylabel("true")
    for i in range(5):
        for j in range(5):
            axes[0, 1].text(j, i, f"{cmn[i, j]:.0f}%", ha="center", va="center",
                            color="white" if cmn[i, j] > 50 else "black",
                            fontsize=9)

    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=list(range(5)), zero_division=0)
    x = np.arange(5)
    width = 0.25
    axes[0, 2].bar(x - width, prec, width, label="precision", color="#3498db")
    axes[0, 2].bar(x,         rec,  width, label="recall",    color="#2ecc71")
    axes[0, 2].bar(x + width, f1,   width, label="F1",        color="#e74c3c")
    axes[0, 2].set_xticks(x)
    axes[0, 2].set_xticklabels([f"G{i}" for i in range(5)])
    axes[0, 2].set_ylim(0, 1.05)
    axes[0, 2].legend()
    axes[0, 2].set_title("Per-class metrics", fontweight="bold")

    per_grade_acc = np.diag(cm) / np.clip(cm.sum(axis=1), 1, None)
    axes[1, 0].bar(range(5), per_grade_acc, color=GRADE_COLORS)
    axes[1, 0].set_xticks(range(5))
    axes[1, 0].set_xticklabels([f"G{i}" for i in range(5)])
    axes[1, 0].set_ylim(0, 1.05)
    axes[1, 0].set_ylabel("recall / accuracy")
    axes[1, 0].set_title("Per-grade accuracy", fontweight="bold")

    for g in range(5):
        vals = p_oof[y_true == g]
        if len(vals):
            axes[1, 1].hist(vals, bins=40, alpha=0.5, label=f"true={g}",
                            color=GRADE_COLORS[g])
    for t in thresholds:
        axes[1, 1].axvline(t, color="black", linestyle="--", linewidth=0.8)
    axes[1, 1].set_title("Regression output by true grade", fontweight="bold")
    axes[1, 1].set_xlabel("regression prediction")
    axes[1, 1].legend(fontsize=8)

    diff = np.abs(y_true - y_pred)
    within = [(diff <= k).mean() for k in range(5)]
    axes[1, 2].bar(range(5), within, color="#27ae60")
    axes[1, 2].set_xticks(range(5))
    axes[1, 2].set_xticklabels([f"≤{k}" for k in range(5)])
    axes[1, 2].set_ylim(0, 1.05)
    axes[1, 2].set_title("Cumulative: within-N-grades accuracy",
                         fontweight="bold")
    for i, v in enumerate(within):
        axes[1, 2].text(i, v + 0.02, f"{v * 100:.1f}%",
                        ha="center", fontsize=9)

    plt.suptitle("OOF Ensemble — full metric dashboard",
                 fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(LOG_DIR / "oof_dashboard.png", dpi=120, bbox_inches="tight")
    plt.show()
    print(f"  💾 saved → {LOG_DIR}/oof_dashboard.png")


## 9 · Stage 2 — fine-tune on IDRiD

For each `(backbone, seed)` we pick the **best-fold** Stage-1 checkpoint and refine it on IDRiD with:
- Very low LR (`1e-5`) — we don't want to destroy Stage-1 generalisation.
- Partial unfreeze — keep early backbone frozen, unfreeze only last 2 stages + head.
- 6 epochs with cosine decay, EMA tracking.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 9 — STAGE 2: FINE-TUNE ON IDRiD
# ═══════════════════════════════════════════════════════════════
t0 = step_header(9, "STAGE 2 FINE-TUNE ON IDRiD")

idrid_df = idrid_clean.copy()
if len(idrid_df) == 0:
    print("  ⚠️  No IDRiD data available — skipping Stage 2.")
else:
    print(f"  IDRiD total: {len(idrid_df):,}  "
          f"({idrid_df['split'].value_counts().to_dict()})")

    if "test" in set(idrid_df["split"]):
        idrid_tr = idrid_df[idrid_df["split"] == "train"].reset_index(drop=True)
        idrid_va = idrid_df[idrid_df["split"] == "test"].reset_index(drop=True)
    else:
        idrid_df = idrid_df.sample(frac=1.0, random_state=42).reset_index(drop=True)
        cut = int(len(idrid_df) * 0.85)
        idrid_tr = idrid_df.iloc[:cut].reset_index(drop=True)
        idrid_va = idrid_df.iloc[cut:].reset_index(drop=True)
    print(f"  Stage-2 split — train={len(idrid_tr):,}  val={len(idrid_va):,}")

    for cfg in MODEL_CONFIGS:
        for seed in SEEDS:
            # Pick the best Stage-1 fold for this (backbone, seed)
            best_fold_qwk, best_fold_ckpt = -1, None
            for fold in range(NUM_FOLDS):
                p = CKPT_DIR / f"{cfg['name']}_s{seed}_f{fold}_best.pt"
                if not p.exists():
                    continue
                st = safe_load(p, "cpu")
                if st.get("val_qwk", 0) > best_fold_qwk:
                    best_fold_qwk = st["val_qwk"]
                    best_fold_ckpt = p

            if best_fold_ckpt is None:
                print(f"  ⚠️  no Stage-1 ckpt for {cfg['name']}_s{seed} — skip")
                continue

            s2_tag  = f"s2_{cfg['name']}_s{seed}"
            s2_ckpt = CKPT_DIR / f"{s2_tag}_best.pt"
            if is_done(s2_tag) and s2_ckpt.exists() and not FORCE_STAGE2_RETRAIN:
                print(f"  ✅ {s2_tag}: cached")
                continue

            seed_everything(seed + 999)
            print(f"\n  ▶ {s2_tag}  (from {best_fold_ckpt.name}, "
                  f"QWK={best_fold_qwk:.4f})")

            model = build_model(cfg["backbone"], pretrained=False).to(DEVICE)
            st = safe_load(best_fold_ckpt, DEVICE)
            model.load_state_dict(st["model"], strict=False)
            ema = EMA(model, decay=EMA_DECAY)
            criterion = nn.SmoothL1Loss()

            # Partial unfreeze: keep early backbone frozen, unfreeze late stages
            for n, p in model.backbone.named_parameters():
                p.requires_grad_(False)
            for n, p in model.backbone.named_parameters():
                if any(k in n for k in ("blocks.6", "blocks.5", "stages.3",
                                        "stages.2", "layer4", "layer3")):
                    p.requires_grad_(True)

            tr_ds = DRDataset(idrid_tr, transform=build_train_tf(S2_SIZE))
            va_ds = DRDataset(idrid_va, transform=build_val_tf(S2_SIZE))
            tr_dl = make_loader(tr_ds, S2_BATCH,
                                sampler=build_sampler(idrid_tr), drop_last=True)
            va_dl = make_loader(va_ds, S2_BATCH, shuffle=False)

            optimizer = build_optim(model, S2_LR, WD)
            scheduler = torch.optim.lr_scheduler.LambdaLR(
                optimizer, lambda e: cosine_lr(e, S2_EPOCHS, 1))
            scaler = torch.cuda.amp.GradScaler(enabled=AMP_ENABLED)

            best_qwk = -1.0
            best_thr = None
            for ep in range(S2_EPOCHS):
                t_ep = time.time()
                train_loss = train_one_epoch(model, tr_dl, optimizer, scaler,
                                             criterion, ema, accum_steps=1)
                scheduler.step()

                ema.apply(model)
                vp, vt = predict(model, va_dl)
                ema.restore(model)

                thr, _ = optimize_thresholds(vp, vt.astype(int))
                cur = qwk(vt.astype(int), preds_to_labels(vp, thr))
                acc = accuracy_score(vt.astype(int), preds_to_labels(vp, thr))
                print(f"    ep {ep + 1}/{S2_EPOCHS}  loss={train_loss:.4f}  "
                      f"val_qwk={cur:.4f}  val_acc={acc:.4f}  "
                      f"({time.time() - t_ep:.0f}s)")

                if cur > best_qwk:
                    best_qwk = cur
                    best_thr = thr
                    ema.apply(model)
                    safe_save({
                        "model":      model.state_dict(),
                        "val_qwk":    cur,
                        "thresholds": thr,
                        "backbone":   cfg["backbone"],
                        "size":       S2_SIZE,
                    }, s2_ckpt)
                    ema.restore(model)

            mark_done(s2_tag)
            print(f"    ✓ {s2_tag} best QWK={best_qwk:.4f}")

step_done(t0)


## 10 · External validation on Messidor-2

Messidor-2 was **never** seen during Stage 1 or Stage 2 — this is the real generalisation test. We ensemble every available final checkpoint (preferring Stage-2 over best Stage-1) with TTA (original + hflip + vflip), then apply the OOF-optimised thresholds from Section 8.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 10 — EXTERNAL VALIDATION ON MESSIDOR-2
# ═══════════════════════════════════════════════════════════════
t0 = step_header(10, "EXTERNAL VALIDATION — MESSIDOR-2")

if len(mss_clean) == 0:
    print("  ⚠️  No Messidor-2 data available — skipping.")
else:
    # Gather final-stage checkpoints (prefer Stage 2, fall back to best Stage 1)
    final_ckpts = []
    for cfg in MODEL_CONFIGS:
        for seed in SEEDS:
            s2 = CKPT_DIR / f"s2_{cfg['name']}_s{seed}_best.pt"
            if s2.exists():
                final_ckpts.append((cfg, seed, s2, "s2"))
                continue
            best_q, best_p = -1, None
            for fold in range(NUM_FOLDS):
                p = CKPT_DIR / f"{cfg['name']}_s{seed}_f{fold}_best.pt"
                if not p.exists():
                    continue
                st = safe_load(p, "cpu")
                if st.get("val_qwk", 0) > best_q:
                    best_q, best_p = st["val_qwk"], p
            if best_p is not None:
                final_ckpts.append((cfg, seed, best_p, "s1"))

    print(f"  Models in ensemble: {len(final_ckpts)}")
    for cfg, s, p, stg in final_ckpts:
        print(f"     • {stg}  {cfg['name']}_s{s}  ←  {p.name}")

    def tta_predict(model, df, size, batch=S2_BATCH):
        ds = DRDataset(df, transform=build_val_tf(size))
        dl = make_loader(ds, batch, shuffle=False)
        p_orig, y_true = predict(model, dl)
        if not TTA_FLIPS:
            return p_orig, y_true
        acc = [p_orig]

        @torch.no_grad()
        def _one_flip(flip_code):
            model.eval()
            out = []
            for batch_ in tqdm(dl, desc=f"TTA flip={flip_code}", leave=False):
                if isinstance(batch_, (list, tuple)):
                    x, _ = batch_
                else:
                    x = batch_
                x = x.to(DEVICE, non_blocking=True)
                if flip_code == "h":
                    x = torch.flip(x, dims=[-1])
                elif flip_code == "v":
                    x = torch.flip(x, dims=[-2])
                with torch.cuda.amp.autocast(enabled=AMP_ENABLED):
                    out.append(model(x).float().cpu().numpy())
            return np.concatenate(out)

        acc.append(_one_flip("h"))
        acc.append(_one_flip("v"))
        return np.mean(acc, axis=0), y_true

    all_preds = []
    y_true_final = None
    for cfg, seed, ckpt_path, stg in final_ckpts:
        st = safe_load(ckpt_path, DEVICE)
        model = build_model(cfg["backbone"], pretrained=False).to(DEVICE)
        model.load_state_dict(st["model"], strict=False)
        size = st.get("size", S2_SIZE)
        p_, y_ = tta_predict(model, mss_clean, size)
        all_preds.append(p_)
        y_true_final = y_
        del model
        gc.collect()
        if DEVICE_KIND == "cuda":
            torch.cuda.empty_cache()

    mean_preds = np.mean(all_preds, axis=0)

    thr_path = EXPORT_DIR / "thresholds.json"
    if thr_path.exists():
        thresholds = json.loads(thr_path.read_text())["thresholds"]
    else:
        thresholds, _ = optimize_thresholds(mean_preds, y_true_final.astype(int))

    labels_pred = preds_to_labels(mean_preds, thresholds)
    y_int = y_true_final.astype(int)

    print("\n  ─── Messidor-2 external metrics ───")
    print(f"    QWK       : {qwk(y_int, labels_pred):.4f}")
    print(f"    Accuracy  : {accuracy_score(y_int, labels_pred):.4f}")
    print(f"    F1 (macro): {f1_score(y_int, labels_pred, average='macro', zero_division=0):.4f}")

    np.save(OOF_DIR / "messidor_preds.npy", mean_preds)
    np.save(OOF_DIR / "messidor_true.npy",  y_int)

    cm = confusion_matrix(y_int, labels_pred, labels=list(range(NUM_CLASSES)))
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Greens",
                xticklabels=range(NUM_CLASSES),
                yticklabels=range(NUM_CLASSES))
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(f"Messidor-2 external CM  (QWK={qwk(y_int, labels_pred):.3f})")
    plt.tight_layout()
    plt.savefig(LOG_DIR / "messidor_cm.png", dpi=120)
    plt.show()

step_done(t0)


## 11 · Probability calibration

The model outputs a single continuous regression score. For downstream consumers that want class probabilities (e.g. the Streamlit UI), we fit a truncated Gaussian centred at the prediction and sweep `σ ∈ [0.25, 1.25]` to minimise OOF Brier score. The chosen `σ` ships in `calibration.json`.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 11 — SOFT PROBABILITIES + CALIBRATION CURVE
# ═══════════════════════════════════════════════════════════════
def regression_to_probs(preds, sigma=0.5, num_classes=5):
    """Continuous preds → class probabilities via truncated Gaussians."""
    preds = np.asarray(preds).reshape(-1, 1)
    classes = np.arange(num_classes).reshape(1, -1)
    logp = -0.5 * ((preds - classes) / sigma) ** 2
    logp -= logp.max(axis=1, keepdims=True)
    p = np.exp(logp)
    p /= p.sum(axis=1, keepdims=True)
    return p


t0 = step_header(11, "CALIBRATION")

if (OOF_DIR / "ensemble_preds.npy").exists():
    p_oof = np.load(OOF_DIR / "ensemble_preds.npy")
    y_oof = np.load(OOF_DIR / "ensemble_true.npy")

    best_sigma, best_brier = 0.5, 1e9
    for sg in np.arange(0.25, 1.25, 0.05):
        probs = regression_to_probs(p_oof, sigma=sg, num_classes=NUM_CLASSES)
        y_oh = np.eye(NUM_CLASSES)[y_oof]
        brier = float(((probs - y_oh) ** 2).sum(axis=1).mean())
        if brier < best_brier:
            best_brier, best_sigma = brier, sg
    print(f"  OOF-calibrated σ = {best_sigma:.2f}   Brier = {best_brier:.4f}")

    probs = regression_to_probs(p_oof, sigma=best_sigma,
                                num_classes=NUM_CLASSES)
    plt.figure(figsize=(5, 5))
    for k in range(NUM_CLASSES):
        y_bin = (y_oof == k).astype(int)
        frac_pos, mean_pred = calibration_curve(
            y_bin, probs[:, k], n_bins=10, strategy="uniform")
        plt.plot(mean_pred, frac_pos, marker="o", label=f"class {k}")
    plt.plot([0, 1], [0, 1], "k--", alpha=0.5)
    plt.xlabel("Predicted probability")
    plt.ylabel("Empirical frequency")
    plt.title(f"OOF reliability diagram  (σ={best_sigma:.2f})")
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(LOG_DIR / "calibration_curve.png", dpi=120)
    plt.show()

    (EXPORT_DIR / "calibration.json").write_text(json.dumps({
        "sigma":     float(best_sigma),
        "brier_oof": best_brier,
    }, indent=2))
    print(f"  💾 saved → {EXPORT_DIR / 'calibration.json'}")
else:
    print("  ⚠️  OOF arrays not found — run Section 8 first.")

step_done(t0)


## 12 · Inference pipeline + Advanced Explainability (EigenCAM + Score-CAM)

End-to-end: single image → grade + probabilities + lesion heatmap.

**`infer(path)`** — loads every final model once, runs TTA, ensembles the 6 predictions, maps through thresholds, and returns class probabilities via the calibrated σ.

**`make_explainability_figure(model, backbone, path)`** — EigenCAM + Score-CAM on the regression output, consensus lesion segmentation overlay, and a 6-panel figure.

### 12.1 · Single-image inference

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 12.1 — SINGLE-IMAGE INFERENCE (with TTA)
# ═══════════════════════════════════════════════════════════════
_INF_MODELS = None


def _load_inference_models():
    """Load every final model (Stage-2 preferred, Stage-1 fallback)."""
    global _INF_MODELS
    if _INF_MODELS is not None:
        return _INF_MODELS
    models = []
    for cfg in MODEL_CONFIGS:
        for seed in SEEDS:
            s2 = CKPT_DIR / f"s2_{cfg['name']}_s{seed}_best.pt"
            pth = s2 if s2.exists() else None
            if pth is None:
                best_q, best_p = -1, None
                for fold in range(NUM_FOLDS):
                    p = CKPT_DIR / f"{cfg['name']}_s{seed}_f{fold}_best.pt"
                    if not p.exists():
                        continue
                    st = safe_load(p, "cpu")
                    if st.get("val_qwk", 0) > best_q:
                        best_q, best_p = st["val_qwk"], p
                pth = best_p
            if pth is None:
                continue
            st = safe_load(pth, DEVICE)
            m = build_model(cfg["backbone"], pretrained=False).to(DEVICE)
            m.load_state_dict(st["model"], strict=False)
            m.eval()
            models.append((cfg, seed, m, st.get("size", S2_SIZE)))
    _INF_MODELS = models
    return models


@torch.no_grad()
def infer(img_path: str, apply_tta: bool = True) -> dict:
    """Predict grade + probabilities for a single image."""
    models = _load_inference_models()
    if not models:
        return {"error": "no trained models found"}
    img = cv2.imread(str(img_path))
    if img is None:
        return {"error": f"cannot read {img_path}"}
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    preds = []
    for cfg, seed, m, size in models:
        tf = build_val_tf(size)
        x = tf(image=img_rgb)["image"].unsqueeze(0).to(DEVICE)
        with torch.cuda.amp.autocast(enabled=AMP_ENABLED):
            p0 = float(m(x).item())
        if apply_tta:
            with torch.cuda.amp.autocast(enabled=AMP_ENABLED):
                ph = float(m(torch.flip(x, dims=[-1])).item())
                pv = float(m(torch.flip(x, dims=[-2])).item())
            preds.append(np.mean([p0, ph, pv]))
        else:
            preds.append(p0)
    ens = max(0.0, min(4.0, float(np.mean(preds))))

    thr_path = EXPORT_DIR / "thresholds.json"
    thr = json.loads(thr_path.read_text())["thresholds"] if thr_path.exists() \
        else [0.5, 1.5, 2.5, 3.5]
    grade = int(preds_to_labels(np.array([ens]), thr)[0])

    cal_path = EXPORT_DIR / "calibration.json"
    sigma = float(json.loads(cal_path.read_text())["sigma"]) if cal_path.exists() \
        else 0.5
    probs = regression_to_probs(np.array([ens]), sigma=sigma,
                                num_classes=NUM_CLASSES)[0]
    return {
        "grade":            grade,
        "regression_score": ens,
        "probabilities":    {int(k): float(v) for k, v in enumerate(probs)},
        "thresholds":       thr,
        "sigma":            sigma,
    }


print("✅ Inference helper ready. Example:")
print("   result = infer(r'R:/Dataset/aptos2019/test_images/0005cfc8afb6.png')")
print("   print(result)")


### 12.2 · Advanced explainability — EigenCAM + Score-CAM

Replaces Grad-CAM++ with two complementary gradient-free methods that are more robust and faithful to the model's true decision process:

- **EigenCAM** — SVD of spatial activation maps; fast, gradient-free, numerically stable.
- **Score-CAM** — perturbation-based channel weighting; no backprop, highest faithfulness.

Output: 6-panel figure — Original | EigenCAM | EigenCAM overlay | Score-CAM | Score-CAM overlay | Consensus lesion mask.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# STEP 12.2 — ADVANCED EXPLAINABILITY: EigenCAM + Score-CAM
# -----------------------------------------------------------------------
# Replaces the basic Grad-CAM++ with two complementary,
# industry-grade activation mapping methods:
#
#   ① EigenCAM (gradient-free)
#      Computes the first principal component (SVD) of the spatial
#      feature maps at a target layer. Avoids all gradient-flow issues
#      (vanishing/exploding grads, dead ReLU paths).  Stable, fast.
#      Reference: Muhammad & Yeasin, EigenCAM, 2020.
#
#   ② Score-CAM (perturbation-based)
#      Generates a unique upsampled mask from each activation channel,
#      forward-passes the masked image, and weights channels by the
#      resulting confidence drop. No backprop required — more faithful
#      to the true input-output relationship.
#      Reference: Wang et al., Score-CAM, CVPR 2020.
#
# Both methods are implemented without external dependencies beyond
# standard PyTorch + OpenCV.
#
# Output: 6-panel figure
#   Original | EigenCAM | EigenCAM overlay |
#   Score-CAM | Score-CAM overlay | Lesion mask (top-20 %)
# ═══════════════════════════════════════════════════════════════════════


# ── Utility: resolve target layer for any supported backbone ──────────
def _resolve_target_layer(model: nn.Module, backbone_name: str) -> nn.Module:
    """
    Heuristically find the last convolutional feature-producing layer.
    Works for EfficientNetV2-S, EfficientNet-B3 (timm), SE-ResNeXt26t.
    Falls back to the last conv-bearing child module.
    """
    bb = model.backbone
    name_lower = backbone_name.lower()

    # EfficientNet family (timm): last pointwise conv before head
    if "efficientnet" in name_lower:
        for attr in ("conv_head", "conv_pw", "blocks"):
            if hasattr(bb, attr):
                layer = getattr(bb, attr)
                if not isinstance(layer, nn.Sequential):
                    return layer
                # walk sequential backward to find last conv block
                for blk in reversed(list(layer.children())):
                    sub_convs = [m for m in blk.modules()
                                 if isinstance(m, nn.Conv2d)]
                    if sub_convs:
                        return sub_convs[-1]

    # SE-ResNeXt / ResNet family (timm): last residual stage
    for attr in ("layer4", "stages"):
        if hasattr(bb, attr):
            stage = getattr(bb, attr)
            last_block = list(stage.children())[-1] if hasattr(stage, "__iter__") else stage
            convs = [m for m in last_block.modules() if isinstance(m, nn.Conv2d)]
            if convs:
                return convs[-1]

    # Generic fallback: deepest Conv2d in the backbone
    all_convs = [m for m in bb.modules() if isinstance(m, nn.Conv2d)]
    if all_convs:
        return all_convs[-1]
    raise RuntimeError(f"Cannot resolve target layer for backbone: {backbone_name}")


# ── EigenCAM ──────────────────────────────────────────────────────────
class EigenCAM:
    """
    Gradient-free CAM via principal component of spatial feature maps.
    Computes SVD of the (C × H·W) activation matrix; the first right
    singular vector gives the dominant spatial activation direction.
    """

    def __init__(self, model: nn.Module, target_layer: nn.Module):
        self.model        = model.eval()
        self.target_layer = target_layer
        self._activations = None
        self._hook        = target_layer.register_forward_hook(self._save_act)

    def _save_act(self, module, inp, out):
        self._activations = out.detach().float()   # (1, C, H, W)

    def remove_hooks(self):
        self._hook.remove()

    @torch.no_grad()
    def __call__(self, x: torch.Tensor) -> np.ndarray:
        """
        Returns CAM of shape (H_img, W_img), normalised to [0, 1].
        x : (1, 3, H, W) preprocessed tensor on the model's device.
        """
        _ = self.model(x)
        act = self._activations[0]   # (C, H, W)
        C, H, W = act.shape
        # Reshape → (C, H*W), zero-centre per channel, then SVD
        flat = act.view(C, -1)
        flat = flat - flat.mean(dim=1, keepdim=True)
        try:
            _, _, Vt = torch.linalg.svd(flat, full_matrices=False)
            principal = Vt[0].view(H, W)           # first principal component
        except Exception:
            # Numerical fallback: channel-mean
            principal = flat.mean(0).view(H, W)
        cam = F.relu(principal).cpu().numpy()
        cam -= cam.min()
        denom = cam.max() + 1e-6
        return cam / denom


# ── Score-CAM ─────────────────────────────────────────────────────────
class ScoreCAM:
    """
    Perturbation-based CAM: each activation channel masks the input
    image; the softmax-equivalent output drop weights each channel.
    Gradient-free — robust across architectures and loss types.

    `max_channels`: cap on number of activation channels to perturb
    (speed/quality trade-off; 64 is a good default for 4 GB VRAM).
    """

    def __init__(self, model: nn.Module, target_layer: nn.Module,
                 max_channels: int = 64):
        self.model        = model.eval()
        self.target_layer = target_layer
        self.max_channels = max_channels
        self._activations = None
        self._hook        = target_layer.register_forward_hook(self._save_act)

    def _save_act(self, module, inp, out):
        self._activations = out.detach().float()

    def remove_hooks(self):
        self._hook.remove()

    @torch.no_grad()
    def __call__(self, x: torch.Tensor) -> np.ndarray:
        """
        Returns Score-CAM of shape (H_img, W_img), normalised to [0, 1].
        x : (1, 3, H, W) preprocessed tensor on the model's device.
        """
        _ = self.model(x)
        act = self._activations[0]   # (C, H, W)
        C, fH, fW = act.shape
        iH, iW    = x.shape[2], x.shape[3]

        # Sample channels if C > max_channels (preserve highest-energy ones)
        if C > self.max_channels:
            channel_energy = act.pow(2).sum(dim=(1, 2))
            top_idx = torch.argsort(channel_energy, descending=True)[:self.max_channels]
            act = act[top_idx]
            C   = self.max_channels

        # Baseline score (unmasked input)
        baseline = float(self.model(x).squeeze())

        # Per-channel masked scores
        scores = []
        for ch in range(C):
            mask = act[ch]
            mask = (mask - mask.min()) / (mask.max() - mask.min() + 1e-6)  # [0,1]
            # Upsample mask to input resolution
            mask_up = F.interpolate(
                mask.unsqueeze(0).unsqueeze(0),
                size=(iH, iW), mode="bilinear", align_corners=False
            ).squeeze()                                                     # (iH, iW)
            x_masked = x * mask_up.unsqueeze(0).unsqueeze(0)               # broadcast
            score_ch  = float(self.model(x_masked).squeeze())
            scores.append(score_ch - baseline)                              # delta

        scores_t = torch.tensor(scores, dtype=torch.float32)
        # Soft-max weights over positive deltas (channels that increase activation)
        weights = F.softmax(F.relu(scores_t), dim=0)                       # (C,)

        # Weighted combination of activation maps
        cam = torch.zeros(fH, fW, dtype=torch.float32)
        for ch in range(C):
            mask = act[ch]
            mask = (mask - mask.min()) / (mask.max() - mask.min() + 1e-6)
            cam += weights[ch] * mask.cpu()

        cam = F.relu(cam).numpy()
        cam -= cam.min()
        denom = cam.max() + 1e-6
        return cam / denom


# ── Master visualisation routine ──────────────────────────────────────
def make_explainability_figure(
    model: nn.Module,
    backbone_name: str,
    image_path: str,
    img_size: int = 320,
    score_cam_channels: int = 64,
    lesion_percentile: float = 80.0,
    save_path=None,
) -> dict:
    """
    Generate a 6-panel explainability figure for a single fundus image.

    Panels
    ------
    1. Original image
    2. EigenCAM heatmap
    3. EigenCAM overlay
    4. Score-CAM heatmap
    5. Score-CAM overlay
    6. Consensus lesion mask (top-N% of element-wise max)

    Returns
    -------
    dict with keys: img_rgb, eigen_cam, score_cam, overlay_eigen,
                    overlay_score, lesion_mask, regression_score
    """
    # Load image
    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        raise FileNotFoundError(f"Cannot read: {image_path}")
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    H_orig, W_orig = img_rgb.shape[:2]

    # Pre-process for inference
    tf = build_val_tf(img_size)
    x  = tf(image=img_rgb)["image"].unsqueeze(0).to(DEVICE)

    # Resolve target layer
    target_layer = _resolve_target_layer(model, backbone_name)
    print(f"  Target layer : {type(target_layer).__name__} "
          f"[in {backbone_name}]")

    model.eval()

    # ① EigenCAM
    eigen_engine = EigenCAM(model, target_layer)
    with torch.no_grad():
        eigen_cam_raw = eigen_engine(x)
    eigen_engine.remove_hooks()
    eigen_cam = cv2.resize(eigen_cam_raw, (W_orig, H_orig))

    # ② Score-CAM  (forward-only, slower — progress printed)
    print(f"  Running Score-CAM (max_channels={score_cam_channels}) …")
    score_engine  = ScoreCAM(model, target_layer, max_channels=score_cam_channels)
    score_cam_raw = score_engine(x)
    score_engine.remove_hooks()
    score_cam = cv2.resize(score_cam_raw, (W_orig, H_orig))

    # Regression score (no grad needed)
    with torch.no_grad():
        reg_score = float(model(x).squeeze().item())

    # ── Colour overlays ───────────────────────────────────────────────
    def _heatmap_overlay(img: np.ndarray, cam: np.ndarray,
                         alpha: float = 0.45) -> np.ndarray:
        heat = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_TURBO)
        heat = cv2.cvtColor(heat, cv2.COLOR_BGR2RGB)
        return cv2.addWeighted(img, 1 - alpha, heat, alpha, 0)

    overlay_eigen = _heatmap_overlay(img_rgb, eigen_cam)
    overlay_score = _heatmap_overlay(img_rgb, score_cam)

    # ── Consensus lesion mask ─────────────────────────────────────────
    consensus = np.maximum(eigen_cam, score_cam)
    thr_val   = np.percentile(consensus, lesion_percentile)
    lesion_mask = (consensus >= thr_val).astype(np.uint8)
    # Morphological clean-up: close small holes
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    lesion_mask = cv2.morphologyEx(lesion_mask, cv2.MORPH_CLOSE, k)

    lesion_vis = img_rgb.copy()
    lesion_vis[lesion_mask > 0] = (
        lesion_vis[lesion_mask > 0] * 0.35 + np.array([255, 50, 50]) * 0.65
    ).clip(0, 255).astype(np.uint8)

    # ── 6-panel figure ────────────────────────────────────────────────
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.patch.set_facecolor("#0d0d0d")

    panels = [
        (img_rgb,         "Original",                        None),
        (eigen_cam,       f"EigenCAM  (reg={reg_score:.2f})", "turbo"),
        (overlay_eigen,   "EigenCAM overlay",                 None),
        (score_cam,       "Score-CAM",                        "turbo"),
        (overlay_score,   "Score-CAM overlay",                None),
        (lesion_vis,      f"Consensus lesion mask (top {100 - lesion_percentile:.0f}%)", None),
    ]
    for ax, (data, title, cmap) in zip(axes.flatten(), panels):
        ax.imshow(data, cmap=cmap)
        ax.set_title(title, color="white", fontsize=11, fontweight="bold", pad=6)
        ax.axis("off")
        for spine in ax.spines.values():
            spine.set_edgecolor("#444")

    plt.suptitle(
        f"Fundus Explainability — EigenCAM + Score-CAM  |  "
        f"Regression score: {reg_score:.3f}",
        color="white", fontsize=13, fontweight="bold", y=1.01,
    )
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=130, bbox_inches="tight",
                    facecolor=fig.get_facecolor())
        print(f"  💾 saved → {save_path}")
    plt.show()

    return {
        "img_rgb":          img_rgb,
        "eigen_cam":        eigen_cam,
        "score_cam":        score_cam,
        "overlay_eigen":    overlay_eigen,
        "overlay_score":    overlay_score,
        "lesion_mask":      lesion_mask,
        "regression_score": reg_score,
    }


# ── Update inference-model loader compatibility ───────────────────────
# `build_val_tf` must exist (defined in §5.1) before calling the function above.

print("✅ Advanced explainability engine loaded:")
print("   • EigenCAM  — gradient-free, SVD of feature maps")
print("   • Score-CAM — perturbation-based, most faithful to model output")
print()
print("Usage example:")
print("   models = _load_inference_models()")
print("   cfg, seed, model, size = models[0]")
print("   result = make_explainability_figure(")
print("       model, cfg['backbone'],")
print("       image_path=df_ap_c.iloc[0]['img_path'],")
print("       img_size=size,")
print("       save_path=LOG_DIR / 'explainability_sample.png'")
print("   )")


## 13 · Export final models + metadata

Copies every final checkpoint to `EXPORT_DIR/models/` and writes a single `pipeline_metadata.json` capturing config, per-model info, and OOF + external metrics.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 13 — EXPORT FINAL MODELS + METADATA
# ═══════════════════════════════════════════════════════════════
t0 = step_header(13, "EXPORT")

MODELS_OUT = EXPORT_DIR / "models"
MODELS_OUT.mkdir(parents=True, exist_ok=True)

final_ckpts_meta = []
for cfg in MODEL_CONFIGS:
    for seed in SEEDS:
        s2 = CKPT_DIR / f"s2_{cfg['name']}_s{seed}_best.pt"
        if s2.exists():
            src, stage = s2, "s2"
        else:
            best_q, best_p = -1, None
            for fold in range(NUM_FOLDS):
                p = CKPT_DIR / f"{cfg['name']}_s{seed}_f{fold}_best.pt"
                if not p.exists():
                    continue
                st = safe_load(p, "cpu")
                if st.get("val_qwk", 0) > best_q:
                    best_q, best_p = st["val_qwk"], p
            if best_p is None:
                continue
            src, stage = best_p, "s1"

        dst = MODELS_OUT / f"{cfg['name']}_s{seed}_best.pt"
        shutil.copy2(src, dst)
        st = safe_load(src, "cpu")
        final_ckpts_meta.append({
            "name":        cfg["name"],
            "backbone":    cfg["backbone"],
            "seed":        seed,
            "stage":       stage,
            "source":      src.name,
            "exported_to": dst.name,
            "val_qwk":     float(st.get("val_qwk", 0)),
        })

meta = {
    "pipeline_version": "v24",
    "timestamp_utc":    time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "config": {
        "num_folds":       NUM_FOLDS,
        "num_classes":     NUM_CLASSES,
        "seeds":           SEEDS,
        "backbones":       [m["backbone"] for m in MODEL_CONFIGS],
        "phases":          PHASES,
        "s2_epochs":       S2_EPOCHS,
        "s2_lr":           S2_LR,
        "tta":             bool(TTA_FLIPS),
        "mixed_precision": bool(AMP_ENABLED),
        "ema_decay":       EMA_DECAY,
    },
    "final_checkpoints": final_ckpts_meta,
}

if (OOF_DIR / "ensemble_preds.npy").exists():
    p_oof = np.load(OOF_DIR / "ensemble_preds.npy")
    y_oof = np.load(OOF_DIR / "ensemble_true.npy")
    thr = json.loads((EXPORT_DIR / "thresholds.json").read_text())["thresholds"]
    m = eval_regression(p_oof, y_oof, thresholds=thr)
    meta["oof_metrics"] = {k: (v.tolist() if hasattr(v, "tolist") else v)
                           for k, v in m.items() if k != "labels"}

if (OOF_DIR / "messidor_preds.npy").exists():
    pm = np.load(OOF_DIR / "messidor_preds.npy")
    ym = np.load(OOF_DIR / "messidor_true.npy")
    thr = json.loads((EXPORT_DIR / "thresholds.json").read_text())["thresholds"]
    m = eval_regression(pm, ym, thresholds=thr)
    meta["messidor2_metrics"] = {k: (v.tolist() if hasattr(v, "tolist") else v)
                                 for k, v in m.items() if k != "labels"}

(EXPORT_DIR / "pipeline_metadata.json").write_text(
    json.dumps(meta, indent=2, default=str))
print(f"  ✅ Exported {len(final_ckpts_meta)} models → {MODELS_OUT}")
print(f"  ✅ Metadata → {EXPORT_DIR / 'pipeline_metadata.json'}")
step_done(t0)


## 14 · Streamlit deployment app

Run the two cells below in order:

1. **Cell 14.1** — installs Streamlit if missing, defines and writes the app to `WORK_ROOT/app.py`, then launches it as a background subprocess directly from this notebook.
2. **Cell 14.2** — stop the server when you are done.

The app gates every user-uploaded image through the trained fundus validator before allowing a grading prediction. Non-fundus images are rejected with a clear error message.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# STEP 14.1 — LAUNCH STREAMLIT APP FROM NOTEBOOK
# -----------------------------------------------------------------------
# Defines the complete Streamlit application inline, writes it to
# WORK_ROOT/app.py, then starts the server as a background subprocess.
# No manual shell command needed — just run this cell.
# ═══════════════════════════════════════════════════════════════════════
import subprocess, sys, socket, time, textwrap
from pathlib import Path
from IPython.display import display, HTML

# ── Ensure streamlit is installed ─────────────────────────────────────
try:
    import streamlit as _st_check  # noqa: F401
except ImportError:
    print("Installing streamlit …")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "streamlit"],
                   check=True)
    print("✅ streamlit installed")

# ── App source (defined inline, written to disk) ──────────────────────
APP_SOURCE = textwrap.dedent("""
    import sys, json, time
    from pathlib import Path

    import cv2
    import numpy as np
    import streamlit as st
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.nn.parameter import Parameter
    import timm

    # ── Paths (resolved relative to this script) ──────────────────────
    HERE           = Path(__file__).parent
    EXPORT_DIR     = HERE / "export"
    MODELS_DIR     = EXPORT_DIR / "models"
    VALIDATOR_CKPT = EXPORT_DIR / "fundus_validator.pt"
    THRESHOLDS_F   = EXPORT_DIR / "thresholds.json"
    CALIBRATION_F  = EXPORT_DIR / "calibration.json"

    DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    GRADE_NAMES = ["0 — No DR", "1 — Mild", "2 — Moderate",
                   "3 — Severe", "4 — Proliferative DR"]
    MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

    # ── Model definitions (must match training) ────────────────────────
    class GeM(nn.Module):
        def __init__(self, p=3.0, eps=1e-6):
            super().__init__()
            self.p   = Parameter(torch.ones(1) * p)
            self.eps = eps
        def forward(self, x):
            return F.avg_pool2d(
                x.clamp(min=self.eps).pow(self.p),
                (x.size(-2), x.size(-1))
            ).pow(1.0 / self.p)

    class DRModel(nn.Module):
        def __init__(self, backbone_name, drop=0.5):
            super().__init__()
            self.backbone = timm.create_model(
                backbone_name, pretrained=False, num_classes=0, global_pool="")
            feat = self.backbone.num_features
            self.pool = GeM()
            self.head = nn.Sequential(
                nn.Flatten(),
                nn.Linear(feat, 256), nn.BatchNorm1d(256),
                nn.ReLU(True), nn.Dropout(drop), nn.Linear(256, 1)
            )
        def forward(self, x):
            f = self.backbone(x)
            p = f.unsqueeze(-1).unsqueeze(-1) if f.dim() == 2 else self.pool(f)
            return self.head(p).squeeze(-1)

    # ── Pre-processing ─────────────────────────────────────────────────
    def preprocess(img_rgb: np.ndarray, size: int) -> torch.Tensor:
        h, w = img_rgb.shape[:2]
        s    = size / max(h, w)
        img  = cv2.resize(img_rgb, (max(1, int(w * s)), max(1, int(h * s))),
                          interpolation=cv2.INTER_AREA)
        canvas         = np.zeros((size, size, 3), dtype=np.uint8)
        y0, x0         = (size - img.shape[0]) // 2, (size - img.shape[1]) // 2
        canvas[y0:y0 + img.shape[0], x0:x0 + img.shape[1]] = img
        x = (canvas.astype(np.float32) / 255.0 - MEAN) / STD
        return torch.from_numpy(x.transpose(2, 0, 1)).unsqueeze(0).float().to(DEVICE)

    # ── Calibration helper ─────────────────────────────────────────────
    def regression_to_probs(score: float, sigma: float, n: int = 5) -> np.ndarray:
        s   = np.array([[score]])
        cls = np.arange(n).reshape(1, -1)
        lp  = -0.5 * ((s - cls) / sigma) ** 2
        lp -= lp.max(axis=1, keepdims=True)
        p   = np.exp(lp)
        return (p / p.sum(axis=1, keepdims=True))[0]

    # ── Fundus validator: heuristic pre-check ─────────────────────────
    def _fundus_heuristic(img_bgr: np.ndarray) -> bool:
        h, w = img_bgr.shape[:2]
        s    = 256 / max(h, w)
        if s < 1:
            img_bgr = cv2.resize(img_bgr, (int(w * s), int(h * s)))
        gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
        _, mask = cv2.threshold(gray, 10, 255, cv2.THRESH_BINARY)
        fg  = float(mask.mean() / 255.0)
        b, _, r = cv2.split(img_bgr)
        m01 = mask > 0
        rb  = (float(r[m01].mean()) / (float(b[m01].mean()) + 1e-6)
               if m01.sum() > 0 else 0.0)
        return 0.25 <= fg <= 0.95 and rb >= 1.2

    # ── Load all models once per session ──────────────────────────────
    @st.cache_resource(show_spinner="Loading models …")
    def load_all_models():
        thresholds = ([0.5, 1.5, 2.5, 3.5] if not THRESHOLDS_F.exists()
                      else json.loads(THRESHOLDS_F.read_text())["thresholds"])
        sigma = (0.5 if not CALIBRATION_F.exists()
                 else float(json.loads(CALIBRATION_F.read_text())["sigma"]))

        # Fundus validator
        validator = None
        if VALIDATOR_CKPT.exists():
            ck = torch.load(VALIDATOR_CKPT, map_location=DEVICE)
            validator = timm.create_model(
                ck["arch"], pretrained=False, num_classes=1).to(DEVICE).eval()
            validator.load_state_dict(ck["state_dict"])

        # Grading ensemble
        graders = []
        if MODELS_DIR.exists():
            for p in sorted(MODELS_DIR.glob("*.pt")):
                ck = torch.load(p, map_location=DEVICE)
                g  = DRModel(ck.get("backbone", "tf_efficientnetv2_l.in21k_ft_in1k"))
                g.load_state_dict(ck["model"], strict=False)
                graders.append((g.to(DEVICE).eval(), int(ck.get("size", 320))))
        return validator, graders, np.array(thresholds), sigma

    # ── Streamlit UI ───────────────────────────────────────────────────
    st.set_page_config(
        page_title="DR Grading", page_icon="🩺", layout="wide")

    st.title("🩺 Diabetic Retinopathy Grading")
    st.caption(
        "Upload a **fundus photograph** — the pipeline predicts a DR grade (0–4) "
        "with calibrated class probabilities.")

    uploaded = st.file_uploader(
        "Choose a fundus image", type=["png", "jpg", "jpeg", "tif", "tiff"])
    if uploaded is None:
        st.info("Waiting for an image upload …")
        st.stop()

    raw = np.frombuffer(uploaded.read(), dtype=np.uint8)
    img_bgr = cv2.imdecode(raw, cv2.IMREAD_COLOR)
    if img_bgr is None:
        st.error("❌  Could not decode the uploaded file.")
        st.stop()
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    col_img, col_result = st.columns([1, 1])
    col_img.image(img_rgb, caption="Uploaded image", use_column_width=True)

    # ── Step 1: heuristic gate (fast, no model required) ──────────────
    if not _fundus_heuristic(img_bgr):
        col_result.error(
            "❌  This image does not appear to be a fundus photograph "
            "(heuristic check failed — wrong colour profile or disc coverage).")
        st.stop()

    # ── Step 2: learned validator gate ────────────────────────────────
    validator, graders, thresholds, sigma = load_all_models()

    if validator is not None:
        x224  = preprocess(img_rgb, 224)
        with torch.no_grad():
            conf = float(torch.sigmoid(validator(x224).squeeze(-1)).item())
        if conf < 0.5:
            col_result.error(
                f"❌  Fundus validator rejected this image "
                f"(confidence = {conf:.2f} < 0.50). "
                f"Please upload a clear fundus photograph.")
            col_result.progress(conf, text=f"Fundus confidence: {conf:.2f}")
            st.stop()
        col_result.success(f"✅  Fundus confirmed (confidence = {conf:.2f})")
    else:
        col_result.warning("⚠️  Fundus validator checkpoint not found — skipping learned gate.")

    # ── Step 3: ensemble grading ───────────────────────────────────────
    if not graders:
        col_result.error("❌  No grading models found in export/models/. "
                         "Complete training (Sections 7–9) first.")
        st.stop()

    preds = []
    with torch.no_grad():
        for g, size in graders:
            x   = preprocess(img_rgb, size)
            p0  = float(g(x).item())
            ph  = float(g(torch.flip(x, dims=[-1])).item())
            pv  = float(g(torch.flip(x, dims=[-2])).item())
            preds.append(float(np.mean([p0, ph, pv])))

    score = float(np.clip(np.mean(preds), 0.0, 4.0))
    grade = int((score > thresholds).sum())
    probs = regression_to_probs(score, sigma=sigma)

    # ── Results panel ──────────────────────────────────────────────────
    GRADE_COLORS = ["#27ae60", "#f1c40f", "#e67e22", "#e74c3c", "#8e44ad"]

    col_result.markdown(
        f"### Grade **{grade}** — {GRADE_NAMES[grade]}",
        help="0 = No DR  |  1 = Mild  |  2 = Moderate  |  3 = Severe  |  4 = Proliferative")
    col_result.metric("Regression score", f"{score:.3f}", help="Raw model output (0–4 continuous)")

    col_result.markdown("**Class probabilities**")
    for k, (name, prob) in enumerate(zip(GRADE_NAMES, probs)):
        col_result.progress(float(prob), text=f"{name} — {prob * 100:.1f}%")

    st.divider()
    with st.expander("Ensemble details"):
        for idx, p in enumerate(preds):
            st.write(f"Model {idx + 1}: raw score = {p:.4f}")
        st.write(f"Ensemble mean = {score:.4f}  |  Thresholds = {list(thresholds)}")
""")

# ── Write app.py to WORK_ROOT and launch ──────────────────────────────
APP_PATH = WORK_ROOT / "app.py"
APP_PATH.write_text(APP_SOURCE.lstrip("\n"), encoding="utf-8")
print(f"  ✅ App written → {APP_PATH}")

# Free any process already bound to port 8501
def _port_free(port=8501):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(("localhost", port)) != 0

PORT = 8501
if not _port_free(PORT):
    print(f"  ⚠️  Port {PORT} already in use — Streamlit may already be running.")
    display(HTML(
        f'<b>Streamlit is already running at: '        f'<a href="http://localhost:{PORT}" target="_blank">'        f'http://localhost:{PORT}</a></b>'
    ))
else:
    _proc = subprocess.Popen(
        [sys.executable, "-m", "streamlit", "run", str(APP_PATH),
         "--server.port", str(PORT),
         "--server.headless", "true",
         "--server.runOnSave", "false",
         "--browser.gatherUsageStats", "false"],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )

    # Store process handle in notebook global so Cell 14.2 can stop it
    import builtins
    builtins._streamlit_proc = _proc

    # Wait briefly to confirm startup
    time.sleep(3)
    if _proc.poll() is not None:
        out, err = _proc.communicate()
        print("❌  Streamlit failed to start:")
        print(err.decode(errors="replace")[-1000:])
    else:
        print(f"  ✅ Streamlit server started  (PID {_proc.pid})")
        display(HTML(
            f'<div style="padding:10px;background:#1a1a2e;border-radius:8px;">'            f'<span style="color:#e94560;font-size:16px;font-weight:bold;">'            f'🩺 DR Grading App is running</span><br>'            f'<a href="http://localhost:{PORT}" target="_blank" '            f'style="color:#0f3460;font-size:14px;">'            f'▶  Open http://localhost:{PORT}</a>'            f'</div>'
        ))
        print(f"  ▶  http://localhost:{PORT}")
        print(f"  Run Cell 14.2 to stop the server.")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# STEP 14.2 — STOP STREAMLIT SERVER
# Run this cell to cleanly terminate the background Streamlit process.
# ═══════════════════════════════════════════════════════════════════════
import builtins

_proc = getattr(builtins, "_streamlit_proc", None)
if _proc is None:
    print("⚠️  No Streamlit process found in this session.")
elif _proc.poll() is not None:
    print(f"ℹ️  Streamlit process (PID {_proc.pid}) has already exited.")
else:
    _proc.terminate()
    _proc.wait(timeout=5)
    print(f"✅ Streamlit server (PID {_proc.pid}) stopped.")
    builtins._streamlit_proc = None


## 15 · Final pipeline summary

Single printout consolidating OOF + Messidor-2 metrics and listing every shipped artefact.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 15 — FINAL SUMMARY
# ═══════════════════════════════════════════════════════════════
print("═" * 70)
print("  DR Grading Pipeline v24 — Summary")
print("═" * 70)
print(f"  Working dir : {WORK_ROOT}")
print(f"  Checkpoints : {CKPT_DIR}")
print(f"  Export      : {EXPORT_DIR}")
print()

meta_p = EXPORT_DIR / "pipeline_metadata.json"
if meta_p.exists():
    d = json.loads(meta_p.read_text())
    for section in ("oof_metrics", "messidor2_metrics"):
        if section in d:
            print(f"  ── {section} ──")
            for k, v in d[section].items():
                if isinstance(v, list):
                    v = [round(x, 4) for x in v]
                if isinstance(v, float):
                    v = round(v, 4)
                print(f"    {k:<14}: {v}")
            print()

print("  Exported final models:")
for p in sorted((EXPORT_DIR / "models").glob("*.pt")):
    print(f"    • {p.name}  ({p.stat().st_size / 1024**2:.1f} MB)")

print()
print("  ➤  For deployment, ship:")
print("       export/models/")
print("       export/thresholds.json")
print("       export/calibration.json")
print("       export/fundus_validator.pt   (if trained)")
print("       export/streamlit_app.py")
print("═" * 70)


## Optional — soft reset utility

Uncomment inside the cell to wipe `.done` flags (forces re-run) **without** deleting trained weights. Useful after editing config — otherwise cached cells will short-circuit.

In [ ]:
# ── DANGER: soft reset (flags only, keeps trained weights) ─────
# for f in FLAG_DIR.glob("*.done"):
#     f.unlink()
# print("Flags cleared. Re-run the notebook from the top.")
